In [ ]:
%run _bootstrap_dev.ipynb

## Portafoglio da analizzare

In [ ]:
#
# Quale portoflio analizzare
#

portfolio=portfolio_germany_plan
# portfolio=portfolio_alpha_sect
# portfolio=portfolio_alpha_euro
# portfolio=portfolio_alpha_nasdaq100
portfolio = portfolio_alpha_sp100
# portfolio=portfolio_alpha_sect
# portfolio=portfolio_alpha_world
# portfolio = portfolio_alpha_world_vanguard


# Get portfolio data
tickers=portfolio['tickers']

# Supporto a indici: se tickers e' una stringa (ossia non e' una lista predefinita) creo la lista di tickers:
tickers = (
    extract_tickers_from_wikipedia(tickers,exclude=["GOOG"],rename={"BRK.B": "BRK-B"})
    if isinstance(tickers, str)
    else list(tickers)
)

benchmark_portfolio=portfolio['benchmark_portfolio']
benchmark_title=portfolio['benchmark_title']
portfolio_title=portfolio['Title']

# Wfo file save
wfo_results_dir = _TSLAB_DEV_R_WFO_RESULTS_DIR

wfo_file_save=f"{wfo_results_dir}/{portfolio_title}_{year}.wfo_summary.csv"


print(f"\nPortafoglio da analizzare:")
print_dict_kv(portfolio)
print(f"numero di ticker in portfaglio: {BOLD}{len(tickers)}{RESET}")

# Date di analisi
start_date="2015-01-01"
end_date=None
# end_date='2026-01-01'

# anno di selezione
year=2026

print(f"\nPortafogli disponibili:\n{BOLD}{list_available_portfolios()}{RESET}")

print("\nAnalisi intersezioni tickers:")
inter = compute_portfolio_ticker_intersections()
print_dict_kv(inter)

## Download data

In [ ]:
init_cash=100_000
normalize=False # Nei rotazionali va lasciato False -> un titolo non compare nelle selezioni dove ha NaN
lookback_buffer=365

download_start_date = (pd.to_datetime(start_date) - timedelta(days=lookback_buffer)).strftime("%Y-%m-%d")
stocks_data, company_data = fetch_data_and_companies(tickers, download_start_date, end_date,normalize=normalize)
# stocks_data_raw, company_data = fetch_data_and_companies(tickers, download_start_date, end_date,normalize=normalize,auto_adjust=False)
stocks_data_raw = download_data(tickers, download_start_date, end_date, auto_adjust=False)

# stocks_data = stocks_data.ffill()#.bfill().dropna()  # Usa anche bfill per valori iniziali

if benchmark_portfolio:
    print("Creo il benchmark portfolio data")
    benchmark_data = build_benchmark(benchmark_portfolio, stocks_data.index.min(),stocks_data.index.max()).replace(0, np.nan).ffill()
    benchmark_data_raw = build_benchmark(benchmark_portfolio, stocks_data.index.min(),stocks_data.index.max(),auto_adjust=False).replace(0, np.nan).ffill() 
elif benchmark_title:
    benchmark_data = download_data(benchmark_title, stocks_data.index.min(),end_date)
    benchmark_data_raw = download_data(benchmark_title, stocks_data_raw.index.min(),end_date,auto_adjust=False)
else:
    benchmark_data=None

In [ ]:
# Tickers da usare in modalita risk_off
risk_off_data = download_data(risk_off_tickers, download_start_date, end_date)

In [ ]:
display(benchmark_data)
display(stocks_data)
display(stocks_data_raw)
display(risk_off_data)
display(company_data)

## Backtest non ottimizzati

In [ ]:
###### Parametri per il backtest
# rebalance_frequency="ME"
# momentum_lookback_days=126
# riskparity_lookback_days=20
# n_top=3
# momentum_weight=0.7
# filter_ema=True# filter_volatility=True
# filter_min_momentum=True

# %run _bootstrap_dev.ipynb

rebalance_frequency = 'Y' # 
momentum_lookback_days = 60
riskparity_lookback_days = 20
n_top = 8
momentum_weight = 0.8
filter_ema = True
filter_volatility = True
filter_min_momentum = True
use_acceleration =  True

build_other_portfolios = False

# analisi_start_date='2025-01-01'
analisi_start_date=None
# download_start_date = (pd.to_datetime(start_date) - timedelta(days=lookback_buffer)).strftime("%Y-%m-%d")

if analisi_start_date is not None:
    analisi_start_date = pd.Timestamp(analisi_start_date)
    analisi_stocks_data = stocks_data.loc[stocks_data.index >= analisi_start_date - timedelta(days=70)]
    analisi_benchmark_data = benchmark_data.loc[benchmark_data.index >= analisi_start_date]
    
else:
    analisi_stocks_data=stocks_data.copy()
    analisi_benchmark_data=benchmark_data.copy()
    
out = build_rotational_portfolios_vbt(
    stocks_data=analisi_stocks_data,
    benchmark_data=analisi_benchmark_data,
    portfolio_name=f"{portfolio_title} (freq: {rebalance_frequency}) ",
    rebalance_frequency=rebalance_frequency,
    use_acceleration=use_acceleration,
    momentum_lookback_days=momentum_lookback_days,
    riskparity_lookback_days=riskparity_lookback_days,
    n_top=n_top,
    momentum_weight=momentum_weight,
    filter_ema=filter_ema,
    filter_volatility=filter_volatility,
    filter_min_momentum=filter_min_momentum,
    start_date=start_date,
    build_other_portfolios=build_other_portfolios,
    plot=True
)

pf_rot_w = out[0]

if build_other_portfolios:
    pf_mom = out[1]
    pf_rp = out[2]
    pf_bh = out[3]
    sel_tickers_rot_w =  out[4]
    rankings_df = out[5]
    print(f"Totale return: pf_rot_w: {pf_rot_w.total_return():.3f}  pf_mom: {pf_mom.total_return():.3f} pf_rp: {pf_rp.total_return():.3f} pf_bh: {pf_bh.total_return():.3f}\n")
else:
    pf_bh = out[1]
    sel_tickers_rot_w =  out[2]
    rankings_df = out[3]
    print(f"Totale return: pf_rot_w: {pf_rot_w.total_return():.3f} pf_bh: {pf_bh.total_return():.3f}\n")
    
print("Tickers Rotational Weighted:")
display(sel_tickers_rot_w.tail(20))

# metriche: combiniamo in un unico DataFrame
agg_cum = pd.DataFrame({
    "Rotational Weighted": pf_rot_w.cumulative_returns()+1,
    "Benchmark":  pf_bh.cumulative_returns()+1
})

if build_other_portfolios:
    agg_cum["Momentum"]   = pf_mom.cumulative_returns()+1
    agg_cum["RiskParity"] = pf_rp.cumulative_returns()+1
    
df_metrics = analyze_portfolio_metrics(
    port_cumrets=agg_cum,
    portfolio_name=f"{portfolio_title} (freq: {rebalance_frequency}) ",
    benchmark_cumret=pf_bh.cumulative_returns()+1,
    freq="D",
    # sort_by="Sharpe Ratio",
    sort_by='CAGR (%)',
    ascending=False,
    plot_radar=True,
    radar_metrics='all'
)


In [ ]:
# Per confronto con WFO con griglia fissa coincidente coi parametri non ottimizzati.
# 

analysis_start_date="2025-01-01"
analysis_end_date=today().strftime("%Y-%m-%d")


stats = pf_stats_aligned(
    pf=pf_rot_w,
    benchmark=benchmark_data,          # serie FULL
    analysis_start_date=analysis_start_date,  # qui parte il report
    analysis_end_date=analysis_end_date,
    rebase_to=100_000
)
print(f"Statistiche comparate periodo: {BOLD}{analysis_start_date} - {analysis_end_date}{RESET}\n")
print(f"{stats["pf"]["label"]}:")
print_dict_kv(stats["pf"]["metrics"])
print()
print(f"{stats["benchmark"]["label"]}:") 
print_dict_kv(stats["benchmark"]["metrics"])

| Parametro                  | Ruolo                               | Scelta consigliata                   |
| -------------------------- | ----------------------------------- | ------------------------------------ |
| `rebalance_frequency`      | Frequenza operativa                 | `["M", "2M"]` (mensile o bimestrale) |
| `momentum_lookback_days`   | Finestra momentum                   | `[63, 126, 189]` (3, 6, 9 mesi)      |
| `riskparity_lookback_days` | Finestra volatilità                 | `[20, 60]` (1 o 3 mesi)              |
| `n_top`                    | Numero titoli selezionati           | `[4, 6, 8]`                          |
| `momentum_weight`          | Peso del ranking momentum           | `[0.6, 0.7, 0.8]`                    |
| `risk_weight`              | Peso del ranking rischio (1 - mom.) | Derivato automaticamente             |
| `weighting_method`         | Metodo di allocazione               | `["equal", "riskparity"]`            |



## Load WFO Results

In [ ]:
summary_df=load_wfo_summary(wfo_file_save)
summary_df

In [ ]:
analisys_start_date=start_date
analisys_end_date=end_date

pf_rot, pf_benchmark, sel_tickers = build_rotational_portfolios_from_wfo_result( 
    summary_df=summary_df,
    stocks_data=stocks_data,            # Adjusted Data -> Total return
    start_date=analisys_start_date,      # da rivedere la start date 
    end_date=analisys_end_date,
    benchmark_data=benchmark_data,      # Adjusted Data -> Total return
    # stocks_data=stocks_data_raw,        # Raw Data -> Price return
    # benchmark_data=benchmark_data_raw,  # Raw Data -> Price return
    benchmark_title=benchmark_title,
    portfolio_name=f"{portfolio_title} – Real OOS WFO - Total Return",
    # portfolio_name=f"{portfolio_title} – Real OOS WFO - Price Return",
    init_cash=init_cash,
    plot=True,
    debug=False,
)


## Run WFO

In [ ]:
# Esempio di Grid  

param_grid_rotational = {
    # "rebalance_frequency": ["ME", "2ME", "QE"],
    # "rebalance_frequency": ["D","ME","QE"],
    "rebalance_frequency": ["QE","ME"],
    # "rebalance_frequency": ["ME"],
    "momentum_lookback_days": [10,20,40,60],
    # "momentum_lookback_days": [5,10,20],
    "riskparity_lookback_days": [10,20,40,60],
    # "riskparity_lookback_days": [5,10,20],
    "n_top": [5,8,10],
    # "n_top": [1],
    "use_acceleration": [True,False],
    # "use_acceleration": [True],    
    "momentum_weight": [0.5,0.7,1.0],
    # "momentum_weight": [1.0],
    "filter_ema": [True, False],
    # "filter_ema": [True],
    "filter_volatility": [True, False],
     # "filter_volatility": [False],
    "filter_min_momentum": [True, False]
    # "filter_min_momentum": [False]
}

param_grid_rotational_v2 = {
    "rebalance_frequency"      : ["ME", "QE"],
    "momentum_lookback_days"   : [40, 60, 120, 180],
    "riskparity_lookback_days" : [40, 60, 120],
    "n_top"                    : [2, 3, 5, 8],
    "use_acceleration"         : [True, False],
    "momentum_weight"          : [0.5, 0.7, 1.0],
    "filter_ema"               : [True, False],
    "filter_volatility"        : [True, False],
    "filter_min_momentum"      : [True, False],
}
# Combinazioni: 2×4×3×3×2×2×2×2×2 = 1.152

param_grid_rotational_v3 = {
    "rebalance_frequency"      : ["ME", "QE"],
    "momentum_lookback_days"   : [40, 60, 120, 180],
    "riskparity_lookback_days" : [40, 60, 120, 180],
    "n_top"                    : [3, 5, 8],
    "use_acceleration"         : [True, False],
    "momentum_weight"          : [0.5, 0.7, 1.0],
    "filter_ema"               : [True, False],
    "filter_volatility"        : [True, False],
    "filter_min_momentum"      : [True, False],
}
# Combinazioni: 2×4×3×3×2×2×2×2×2 = 1.152

# rebalance_frequency = 'Y' # 'W-FRI'
# momentum_lookback_days = 60
# riskparity_lookback_days = 20
# n_top = 10
# momentum_weight = 0.5
# filter_ema = True
# filter_volatility = True
# filter_min_momentum = True
# use_acceleration =  False

# Top performers anno su anno. Questa e' rotazione semplice, se batte la rotazione WFO intra-anno allora e' meglio questa! 
# Piu' semplice, meno costi!
param_grid_top_performers= {
    "rebalance_frequency": ["YE"],
    "momentum_lookback_days": [10,20,40,60],
    "riskparity_lookback_days": [10,20,40,60],
    "n_top": [5,8,10],
    "use_acceleration": [True,False],
    "momentum_weight": [0.5,0.7,1.0],
    "filter_ema": [True,False],
    "filter_volatility": [True,False],
    "filter_min_momentum": [True,False]
}

param_grid_claude = {
    'momentum_lookback_days': [60, 120],
    'riskparity_lookback_days': [10, 20],
    'n_top': [3, 5, 8],
    'momentum_weight': [0.5, 0.7, 1.0],
    'use_acceleration': [False],
    'filter_ema': [False],
    'filter_volatility': [False],
    'filter_min_momentum': [False]
}

param_grid=param_grid_rotational_v3

# param_grid=param_grid_top_performers
# param_grid=param_grid_claude

# metric="Sharpe Ratio",
metric="CAGR"
plot=False
verbose=False
ratio='3:1'
force_next_year_params=False
cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1

### WFO Standard

In [ ]:
# WFO Standard

pipeline_start_date=None

# pipeline_start_date="2022-01-01"

if pipeline_start_date is not None:
    pipeline_start_date=pd.Timestamp(pipeline_start_date)
else:
    pipeline_start_date=start_date
    
results_std = run_wfo_pipeline(
    # Dati
    stocks_data_raw   = stocks_data_raw,
    stocks_data       = stocks_data,
    benchmark_data    = benchmark_data,
    benchmark_data_raw= benchmark_data_raw,
    tickers           = tickers,
    risk_off_data     = risk_off_data,
    # Parametri WFO
    ratio             = ratio,
    metric            = metric,
    start_date        = pipeline_start_date,
    end_date          = end_date,
    cores             = cores,
    verbose           = verbose,
    force_next_year_params = force_next_year_params,
    use_clustering  = False,
    param_grid      = param_grid,   # obbligatorio se False
    # Parametri portafoglio
    portfolio_title   = portfolio_title,
    benchmark_title   = benchmark_title,
    init_cash         = init_cash,
    risk_on_off       = True,
    plot              = True,
)


In [ ]:
# Accesso ai risultati
pf_rot_std           = results_std['pf_rot']           # con Risk ON/OFF
pf_rot_std_base      = results_std['pf_rot_base']      # senza Risk ON/OFF
regime               = results_std['regime']
summary_df_std       = results_std['summary_df']
sel_tickers_std      = results_std['sel_tickers']      # con Risk ON/OFF
sel_tickers_std_base = results_std['sel_tickers_base'] # senza Risk ON/OFF

In [ ]:
# sel_tickers_std = results_std['sel_tickers']

# (a) la colonna universe c'è ed è popolata
assert 'universe' in sel_tickers_std.columns
assert sel_tickers_std['universe'].apply(len).min() > 0

# # (b) coerenza con n_passed_filters
mismatches = (sel_tickers_std['universe'].apply(len) != sel_tickers_std['n_passed_filters']).sum()
print(f"universe length mismatches: {mismatches}")  # atteso: 0

# # (c) selezioni time-varying (pre-condizione di B2)
# n_unique_selections = sel_tickers_std['tickers'].apply(tuple).nunique()
# print(f"unique selections across rebal_dates: {n_unique_selections}")  # atteso: > 1

In [ ]:

out = generate_rotational_portfolio_performance(
    pf=pf_rot_std,
    portfolio_title=portfolio_title,
    sel_tickers=sel_tickers_std,
    benchmark=benchmark_title,
    benchmark_data=benchmark_data,   # prezzi close
    alpha_analysis=True,
    show_plots=True,
)

In [ ]:

def verify_wfo_portfolio_from_selections(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series | pd.DataFrame,
    sel_tickers_std: pd.DataFrame,
    init_cash: float = 100_000.0,
    rebalance_timing: str = "next_bar",  # "next_bar"  oppure "same_close"
    annualization: int = 365,
):
    """
    Verifica indipendente del portafoglio WFO usando solo:
      - stocks_data
      - benchmark_data
      - sel_tickers_std

    Simula un portafoglio buy-and-hold tra rebalance:
      - alle date di rebalance vende tutto
      - compra equal-weight i ticker selezionati
      - mantiene le quote fino al rebalance successivo

    rebalance_timing:
      - "same_close": entra al close della data di rebalance
      - "next_bar"   : entra alla barra successiva
    """

    # -------------------------
    # Preparazione dati
    # -------------------------
    prices = stocks_data.copy()
    prices.index = pd.to_datetime(prices.index)
    prices = prices.sort_index()

    # fondamentale: vectorbt di solito valorizza su close disponibili;
    # qui evitiamo che NaN finali distruggano il valore della posizione
    prices = prices.ffill()

    if isinstance(benchmark_data, pd.DataFrame):
        bench = benchmark_data.iloc[:, 0].copy()
    else:
        bench = benchmark_data.copy()

    bench.index = pd.to_datetime(bench.index)
    bench = bench.sort_index().ffill()
    bench.name = "Benchmark"

    selections = sel_tickers_std.copy()
    selections.index = pd.to_datetime(selections.index)
    selections = selections.sort_index()

    if "tickers" not in selections.columns:
        raise ValueError("sel_tickers_std deve avere una colonna 'tickers'.")

    # -------------------------
    # Date comuni
    # -------------------------
    common_idx = prices.index.intersection(bench.index)
    prices = prices.loc[common_idx]
    bench = bench.loc[common_idx]

    start = max(prices.index.min(), bench.index.min(), selections.index.min())
    end = min(prices.index.max(), bench.index.max(), selections.index.max())

    prices = prices.loc[start:end]
    bench = bench.loc[start:end]

    if len(prices) == 0:
        raise ValueError("Nessuna data comune tra stocks_data, benchmark_data e sel_tickers_std.")

    # -------------------------
    # Simulazione quote
    # -------------------------
    cash = init_cash
    shares = pd.Series(0.0, index=prices.columns)
    equity = pd.Series(index=prices.index, dtype=float)

    rebal_dates = []

    for rebal_date in selections.index:
        if rebal_date > prices.index[-1]:
            continue

        if rebalance_timing == "same_close":
            pos = prices.index.searchsorted(rebal_date, side="left")
        elif rebalance_timing == "next_bar":
            pos = prices.index.searchsorted(rebal_date, side="right")
        else:
            raise ValueError("rebalance_timing deve essere 'same_close' oppure 'next_bar'.")

        if pos >= len(prices.index):
            continue

        actual_date = prices.index[pos]
        rebal_dates.append(actual_date)

    rebal_map = dict(zip(rebal_dates, selections.loc[selections.index[:len(rebal_dates)], "tickers"]))

    for dt in prices.index:

        px = prices.loc[dt]

        # valore prima di eventuale rebalance
        portfolio_value = cash + (shares * px).sum()

        if dt in rebal_map:
            tickers = rebal_map[dt]

            if isinstance(tickers, str):
                tickers = [
                    x.strip()
                    for x in tickers.replace("[", "").replace("]", "").split(",")
                ]

            tickers = [t for t in tickers if t in prices.columns and pd.notna(px[t]) and px[t] > 0]

            # liquidazione completa
            cash = portfolio_value
            shares[:] = 0.0

            # reinvestimento equal-weight
            if len(tickers) > 0:
                alloc = cash / len(tickers)
                for t in tickers:
                    shares[t] = alloc / px[t]
                cash = 0.0

            portfolio_value = cash + (shares * px).sum()

        equity.loc[dt] = portfolio_value

    # forza start value corretto
    equity.iloc[0] = init_cash

    # -------------------------
    # Benchmark normalizzato
    # -------------------------
    bench_norm = init_cash * bench.loc[equity.index] / bench.loc[equity.index].iloc[0]

    port_rets = equity.pct_change().fillna(0.0)
    bench_rets = bench_norm.pct_change().fillna(0.0)

    # -------------------------
    # Metriche
    # -------------------------
    def max_dd_duration(equity_curve):
        dd = equity_curve / equity_curve.cummax() - 1.0
        in_dd = dd < 0

        max_duration = pd.Timedelta(0)
        start_dd = None

        for date, flag in in_dd.items():
            if flag and start_dd is None:
                start_dd = date
            elif not flag and start_dd is not None:
                max_duration = max(max_duration, date - start_dd)
                start_dd = None

        if start_dd is not None:
            max_duration = max(max_duration, equity_curve.index[-1] - start_dd)

        return max_duration

    def omega_ratio(returns, threshold=0.0):
        excess = returns - threshold
        gains = excess[excess > 0].sum()
        losses = -excess[excess < 0].sum()
        return np.nan if losses == 0 else gains / losses

    n = len(equity)
    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    bench_return = bench_norm.iloc[-1] / bench_norm.iloc[0] - 1

    dd = equity / equity.cummax() - 1
    max_dd = dd.min()

    ann_return = (1 + total_return) ** (annualization / n) - 1

    sharpe = np.nan
    if port_rets.std(ddof=0) != 0:
        sharpe = port_rets.mean() / port_rets.std(ddof=0) * np.sqrt(annualization)

    downside = port_rets[port_rets < 0]
    sortino = np.nan
    if downside.std(ddof=0) != 0:
        sortino = port_rets.mean() / downside.std(ddof=0) * np.sqrt(annualization)

    calmar = np.nan if max_dd == 0 else ann_return / abs(max_dd)

    metrics = pd.Series({
        "Start": equity.index[0],
        "End": equity.index[-1],
        "Period": equity.index[-1] - equity.index[0],
        "Start Value": equity.iloc[0],
        "End Value": equity.iloc[-1],
        "Total Return [%]": total_return * 100,
        "Benchmark Return [%]": bench_return * 100,
        "Max Drawdown [%]": abs(max_dd) * 100,
        "Max Drawdown Duration": max_dd_duration(equity),
        "Sharpe Ratio": sharpe,
        "Calmar Ratio": calmar,
        "Omega Ratio": omega_ratio(port_rets),
        "Sortino Ratio": sortino,
    })

    metrics_df = metrics.to_frame("Value")

    equity_df = pd.DataFrame({
        "Portfolio": equity,
        "Benchmark": bench_norm,
        "Portfolio Returns": port_rets,
        "Benchmark Returns": bench_rets,
    })

    # -------------------------
    # Plot
    # -------------------------
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=equity_df.index,
        y=equity_df["Portfolio"],
        mode="lines",
        name="Portfolio verifica"
    ))

    fig.add_trace(go.Scatter(
        x=equity_df.index,
        y=equity_df["Benchmark"],
        mode="lines",
        name="Benchmark"
    ))

    fig.update_layout(
        title="Verifica indipendente WFO - Portfolio vs Benchmark",
        xaxis_title="Date",
        yaxis_title="Equity",
        hovermode="x unified",
        template="plotly_white",
        height=600,
    )

    return metrics_df, equity_df, fig

In [ ]:
prices_all = pd.concat([stocks_data, risk_off_data], axis=1)
# prices_all = stocks_data


prices_all = prices_all.loc[:, ~prices_all.columns.duplicated()]
prices_all = prices_all.sort_index().ffill()

selected_tickers = sorted({
    t
    for tickers in sel_tickers_std["tickers"]
    for t in tickers
})

missing_tickers = sorted(set(selected_tickers) - set(prices_all.columns))

print("Ticker selezionati:", selected_tickers)
print("Ticker mancanti:", missing_tickers)

print("\nPerformance")

metrics_check_risk, equity_check_risk, fig_check_risk = verify_wfo_portfolio_from_selections(
    # stocks_data=prices_all.loc["2025-02-03":"2026-04-28"],
    # benchmark_data=benchmark_data.loc["2025-02-03":"2026-04-28"],
    stocks_data=prices_all.loc["2024-12-30":],
    benchmark_data=benchmark_data.loc["2024-12-30":],
    # stocks_data=prices_all,
    # benchmark_data=benchmark_data,
    # sel_tickers_std=sel_tickers_std,
    sel_tickers_std=sel_tickers_std,
    rebalance_timing="next_bar",
)

display(metrics_check_risk)
fig_check_risk.show()

#### Save results

In [ ]:
summary_df=summary_df_std
save_rotational_wfo_summary(
    summary_df=summary_df,
    start_date=start_date,
    end_date=end_date,
    file_path=wfo_file_save,
    param_grid=param_grid,
    metric=metric,
    ratio=ratio,
    force_next_year_params=force_next_year_params,
    extra_meta= None
)


#### Show WFO Results

In [ ]:
# short_map = {
#     "rebalance_frequency":    "freq",
#     "momentum_lookback_days": "mom_lb",
#     "riskparity_lookback_days":"rp_lb",
#     "n_top":                  "n_top",
#     "momentum_weight":        "mom_w",
#     "filter_ema":             "f_ema",
#     "filter_volatility":      "f_vol",
#     "filter_min_momentum":    "f_min_m",
#     "Score":                  "score"
# }

# df_sum_comp = summary_df.rename(columns=short_map)
# my_display(title=f"WFO Results {BOLD}{portfolio_title}{RESET} - ({start_date} - {end_date})",data=df_sum_comp)

In [ ]:
# # Solo per debug
# sel_tickers = collect_selections_from_summary(
#     summary_df=summary_df,
#     stocks_data=stocks_data,
#     benchmark_data=benchmark_data,
#     debug=False
# )
# my_display(sel_tickers)

### WFO Clusterizzata ... serve solo per portafogli con universo disomogeneo (es alpha_world) ?

In [ ]:
# # WFO Standard

# results_std = run_wfo_pipeline(
#     # Dati
#     stocks_data_raw   = stocks_data_raw,
#     stocks_data       = stocks_data,
#     benchmark_data    = benchmark_data,
#     benchmark_data_raw= benchmark_data_raw,
#     tickers           = tickers,
#     risk_off_data     = risk_off_data,
#     # Parametri WFO
#     ratio             = ratio,
#     metric            = metric,
#     start_date        = start_date,
#     end_date          = end_date,
#     cores             = cores,
#     verbose           = verbose,
#     force_next_year_params = force_next_year_params,
#     use_clustering  = False,
#     param_grid      = param_grid,   # obbligatorio se False
#     # Parametri portafoglio
#     portfolio_title   = portfolio_title,
#     benchmark_title   = benchmark_title,
#     init_cash         = init_cash,
#     risk_on_off       = True,
#     plot              = True,
# )

use_clustering  = True
adaptive_k = True
adaptive_k_method =  'hybrid' # 'silhouette' | 'corr_threshold' | 'hybrid' (default).
n_clusters      = 5 # valido solo adaptive_k=False
lookback_days   = 504
n_top_min       = 2


results_cluster = run_wfo_pipeline(
    # Dati
    stocks_data_raw   = stocks_data_raw,
    stocks_data       = stocks_data,
    benchmark_data    = benchmark_data,
    benchmark_data_raw= benchmark_data_raw,
    tickers           = tickers,
    risk_off_data     = risk_off_data,
    # Parametri WFO
    ratio             = ratio,
    metric            = metric,
    start_date        = start_date,
    end_date          = end_date,
    cores             = cores,
    verbose           = verbose,
    force_next_year_params = force_next_year_params,
    # Parametri clustering
    use_clustering    = use_clustering,   # default
    n_clusters        = n_clusters,
    adaptive_k        = adaptive_k,
    adaptive_k_method = adaptive_k_method,
    lookback_days     = lookback_days,
    n_top_min         = n_top_min,
    # Parametri portafoglio
    portfolio_title   = portfolio_title,
    benchmark_title   = benchmark_title,
    init_cash         = init_cash,
    risk_on_off       = True,
    plot              = True,
)

In [ ]:
# # Accesso ai risultati
# pf_rot       = results_cluster['pf_rot']        # con Risk ON/OFF
# pf_rot_base  = results_cluster['pf_rot_base']   # senza Risk ON/OFF
# regime       = results_cluster['regime']
# summary_df   = results_cluster['summary_df']
# sel_tickers  = results_cluster['sel_tickers']

# Accesso ai risultati
pf_rot_cluster           = results_cluster['pf_rot']           # con Risk ON/OFF
pf_rot_cluster_base      = results_cluster['pf_rot_base']      # senza Risk ON/OFF
regime               = results_cluster['regime']
summary_df_cluster       = results_cluster['summary_df']
sel_tickers_cluster      = results_cluster['sel_tickers']      # con Risk ON/OFF
sel_tickers_cluster_base = results_cluster['sel_tickers_base'] # senza Risk ON/OFF


#### Save results

## Confronto WFO

In [ ]:
metrics_df = compare_wfo_pipelines(                                                                                            
      results_std     = results_std,                                                                                             
      results_cluster = results_cluster,                                                                                         
      portfolio_title = portfolio_title,                                                                                         
      benchmark_title = benchmark_title,                                                                                         
      plot_radar      = True,                                                                                                    
      # start_date / end_date opzionali per filtrare il periodo
  )              

In [ ]:
# metrics_df = compare_wfo_pipelines(                                                                                            
#       results_std     = results_std,                                                                                             
#       results_cluster = results_cluster,                                                                                         
#       portfolio_title = portfolio_title,                                                                                         
#       benchmark_title = benchmark_title,                                                                                         
#       plot_radar      = True,                                                                                                    
#       # start_date / end_date opzionali per filtrare il periodo
#   )              

In [ ]:
save_rotational_wfo_summary(
    summary_df=summary_df,
    start_date=start_date,
    end_date=end_date,
    file_path=wfo_file_save,
    param_grid=param_grid,
    metric=metric,
    ratio=ratio,
    force_next_year_params=force_next_year_params,
    extra_meta= None
)


## Performance

In [ ]:
# my_display(sel_tickers,title=f"Selezione ticker portfolio  {BOLD}{portfolio_title}{RESET}")

In [ ]:
# my_display(sel_tickers_c,title=f"Selezione ticker (clusterd) portfolio  {BOLD}{portfolio_title}{RESET}")

In [ ]:
#
# Per ricostruire portafoglio con le selezione storiche. I file di selezione (contenenti sel_tikers vengono salvati annu su anno dalla funzione
# r_run_portfolio. Questo perche' i parametri WFO variano di anno in anno ma non e' sufficiente usare queli generati gli anni precedenti. Possono variare anche: universo titoli, 
# motore di selezione, griglia di parametri -> ricostruire le esatte selezioni e' impossibile. Occorre usare quelle salvate anno su anno.
# %run r_functions.ipynb

# Nota per ottenere sel_tickers completo andranno concatenati i file salvati anno su anno (portfolio_{anno}_sel_tickers_current_year.csv)
# analisi_start_date = None
# analisi_start_date= ytd()
# analisi_start_date = '2025-01-01'

# pf_rot, pf_bh, stocks_data, benchmark_data =  build_historical_rotational_portfolios_from_selections(
#     sel_tickers=sel_tickers,
#     benchmark_portfolio=benchmark_portfolio,
#     benchmark_title=benchmark_title,
#     portfolio_name=f"{portfolio_title} – Real OOS WFO - Total Return",
#     init_cash=init_cash,
#     start_date=analisi_start_date,
#     end_date=analisi_end_date,
#     plot=True,
#     # verbose=True
# )


In [ ]:
# # Il benchmark interno e' calcolato da vbt come la BUY & HOLD "equal-cash" (paniere statico). 
# # Non e' ribilanciato giornalmente come si otterrebbe con la media dei rendimenti
# # La funzione lo calcola manualmente per ulteriore verifica.

# # Il confronto col benchmark interno risponde alla domanda: ha senso la rotazione? O guadagno di piu' prendendo tutti i titoli equal weighted? .. se possibile
# # Il confronto col benchmark esterno risponde alla domanda: ha senso usare il portafoglio o mi compro l'indice e ciaone

# res_bh = calc_vbt_internal_benchmark_buyhold_equal_cash(
#     prices_wide=pf_rot.close,     # close effettivo usato da vbt
#     init_cash=init_cash,
#     # start="2025-01-02",
#     # end="2026-01-06",
#     ffill_prices=True
# )

# print("=== Manual benchmark interno (BH equal-cash, NO rebalance) ===")
# print("start_used       :", res_bh["start_used"])
# print("end_used         :", res_bh["end_used"])
# print("n_assets         :", res_bh["n_assets"])
# print("End Value        :", round(res_bh["end_value"], 6))
# print(f"Total Return [%] : {BOLD}{round(res_bh['total_return_pct'], 6)}{RESET}")

# st = pf_rot.stats()
# print("\n=== Portfolio Stats() ===")
# print("Start             :", st.get("Start"))
# print("End               :", st.get("End"))
# print("End Value         :", round(st.get("End Value"),6))
# print("Total Return [%]  :", round(st.get("Total Return [%]"),6))
# print(f"Bench Return [%]  : {BOLD}{round(st.get('Benchmark Return [%]'),6)}{RESET}")


In [ ]:
# results_cluster

In [ ]:
# Ci sono 4 portafogli disponibili:
# pf_rot_std_base        (WFO non clusterizzata)
# pf_rot_std             (WFO non clusterizzata, Risk ON/OFF)
# pf_rot_cluster_base    (WFO clusterizzata)
# pf_rot_cluster         (WFO clusterizzata, Risk ON/OFF)

# E per ognuno di essi 2 possibili confronto con altrettanti benchmark
# Benchmark esterno: definito dal portafoglio (benchmark_data=benchmark_data)
# Benchmark interno: B&H intero universo  (benchmark_data=None)

# Schema decisionale rapido                                                                                                      
                                                                                                                             
# Universo piccolo/omogeneo?
#   ├─ Sì → std_base  (o std se vuoi protezione drawdown)                                                                        
#   └─ No (grande/eterogeneo) → cluster_base  (o cluster se vuoi regime switching)                                               
                                                                                                                             
# Vuoi ridurre il max drawdown?                                                                                                  
#   └─ Sì → aggiungi Risk ON/OFF (std → std_on, cluster → cluster_on)                                                            
                                                                                                                             
# Periodo OOS corto (<3 anni)?
#   └─ Preferisci la versione _base: meno parametri, più robusta                                                                 
                                                                                                                             
# ---
# In pratica, pf_rot_cluster è il più potente ma anche il più fragile se i dati sono pochi. pf_rot_std_base è il più stabile e   
# interpretabile. Gli altri due stanno nel mezzo.                                                                                


out = generate_rotational_portfolio_performance(
    pf=pf_rot_cluster,
    portfolio_title=portfolio_title,
    sel_tickers=sel_tickers_cluster,
    benchmark=benchmark_title,
    benchmark_data=benchmark_data,   # prezzi close
    alpha_analysis=True,
    show_plots=True
)

## Montecarlo (Block Bootstrap) 

In [ ]:
# Approccio #1 -> tengo fisse le selezioni ma mescolo in modo random i return e vedo se i risultati sono stabili. Serve per capire se e' robosta per il timing
%run _bootstrap_dev.ipynb

# Qiuale ptf testare

n_simulations=1_000
block_size=20

portfolio=pf_rot_std
sel_tickers=sel_tickers_std

mc_results = monte_carlo_block_bootstrap_rotational(
    portfolio=portfolio,
    sel_tickers_df=sel_tickers,
    stocks_data=stocks_data,
    n_simulations=n_simulations,
    block_size=block_size,
    random_seed=42
)

# Analizza i risultati
analysis = analyze_mc_results(
    mc_results,
    portfolio,
    confidence_level=0.90,
    print_report=True
)

# Visualizza i risultati
fig = plot_mc_distribution(mc_results, portfolio)


## Montecarlo (Ranking Noise)

In [ ]:
# Approccio #2: applico i parametri WFO su dati inquinati di rumore

# Primo metodo: applico i parametri next year a tutto lo storico. Serve per validare il deploy per il prossimo anno
# Nota: non metodologicamente esatto perche' applica parametri nex year a tutto lo storico, mischiando IS con OOS

n_simulations=1_00
noise_std=0.05

# # Prendi SOLO ultima finestra WFO (parametri per anno prossimo)
params_next_year = summary_df.iloc[-1][:-2].to_dict() # ultima riga (prossimo anno) ed elimino le colonne di Score

# MC su TUTTO lo storico disponibile
# (parametri costanti, dati variano)
mc_results = monte_carlo_ranking_noise(
    stocks_data=stocks_data,  # TUTTO lo storico
    benchmark_data=benchmark_data,
    params=params_next_year,  # parametri fissi per 2026
    n_simulations=n_simulations,
    noise_std=noise_std
)

In [ ]:
# Analizza
analysis = analyze_ranking_noise_results(mc_results, print_report=True)

# Visualizza
plot_ranking_noise_analysis(mc_results, save_path="./ranking_noise.png")

In [ ]:
# Secondo metodo: applico i parametri WFO anno su anno e aggrego  i risultati. Metodologicamente perfetto
%run _bootstrap_dev.ipynb


wfo_mc_results = monte_carlo_wfo_per_window(
    stocks_data=stocks_data,
    benchmark_data=benchmark_data,
    wfo_summary=summary_df, 
    benchmark_title=benchmark_title,
    n_simulations=n_simulations,
    noise_std=noise_std,
    random_seed=42
)

In [ ]:
# Analisi risultati
%run _bootstrap_dev.ipynb

analysis = analyze_wfo_mc_results(wfo_mc_results, print_report=True)

plot_wfo_mc_results(wfo_mc_results, save_path="./wfo_mc_analysis.png")

## Testare la consistenza di un edge 

In [ ]:
# import numpy as np
# import pandas as pd
# from typing import Optional
# import numpy as np
# import pandas as pd
# import statsmodels.api as sm
# import matplotlib.pyplot as plt
# from typing import Optional, Tuple, Dict, Any

# # -----------------------
# # Helpers (reuse your download_data for prices if needed)
# # -----------------------
# def compute_log_returns_from_prices(prices: pd.Series) -> pd.Series:
#     """Log-returns daily (dropna) from a price series."""
#     return np.log(prices).diff().dropna()

# def _ols_coeffs(y: np.ndarray, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
#     """OLS via statsmodels: returns params (const + betas), tstats, pvalues."""
#     Xc = sm.add_constant(X, has_constant='add')
#     model = sm.OLS(y, Xc, missing='drop')
#     res = model.fit()
#     return res.params.values, res.tvalues, res.pvalues

# # -----------------------
# # Main function: rolling regression that accepts vectorbt Portfolio
# # -----------------------
# def rolling_factor_regression_from_pf(
#     pf,
#     factors_df,
#     window=252,
#     annualization=252,
#     plot=True
# ):
#     import statsmodels.api as sm
#     import numpy as np
#     import pandas as pd
#     import matplotlib.pyplot as plt

#     nav = pf.value()
#     if isinstance(nav, pd.DataFrame):
#         nav = nav.iloc[:, 0]

#     asset_rets = np.log(nav).diff().dropna()
#     common_index = asset_rets.index.intersection(factors_df.index)

#     y = asset_rets.reindex(common_index)
#     X_full = factors_df.reindex(common_index)

#     factor_names = list(X_full.columns)

#     results = []

#     for end in range(window, len(common_index) + 1):
#         window_idx = common_index[end - window:end]

#         y_win = y.loc[window_idx]
#         X_win = X_full.loc[window_idx]

#         Xc = sm.add_constant(X_win)

#         model = sm.OLS(y_win, Xc, missing='drop').fit()

#         alpha = model.params.get("const", np.nan)
#         t_alpha = model.tvalues.get("const", np.nan)
#         p_alpha = model.pvalues.get("const", np.nan)

#         row = [window_idx[-1], alpha * annualization, t_alpha, p_alpha]

#         for f in factor_names:
#             row.append(model.params.get(f, np.nan))

#         results.append(row)

#     cols = ["date", "alpha_ann", "t_alpha", "p_alpha"] + [f"beta_{f}" for f in factor_names]
#     rolling_df = pd.DataFrame(results, columns=cols).set_index("date")

#     summary = pd.DataFrame({
#         "alpha_ann_mean": [rolling_df["alpha_ann"].mean()],
#         "alpha_ann_median": [rolling_df["alpha_ann"].median()],
#         "alpha_ann_std": [rolling_df["alpha_ann"].std()],
#         "t_alpha_mean": [rolling_df["t_alpha"].mean()],
#         "pct_positive_alpha": [(rolling_df["alpha_ann"] > 0).mean()],
#         "pct_significant_alpha": [(rolling_df["p_alpha"] < 0.05).mean()]
#     })

#     if plot:
#         plt.figure(figsize=(10,4))
#         plt.plot(rolling_df.index, rolling_df["alpha_ann"], label="Alpha annualizzato")
#         plt.axhline(0, color="black", linestyle="--", linewidth=0.8)
#         plt.title("Rolling Alpha")
#         plt.legend()
#         plt.tight_layout()
#         plt.show()

#     return {"rolling": rolling_df, "summary": summary}    
    
# def build_europe_factors_from_internal_data(
#     stocks_data: pd.DataFrame,
#     company_data: pd.DataFrame,
#     smb_q: float = 0.3,
#     use_logrets: bool = True,
#     mom_formation_days: int = 252,
#     mom_skip_days: int = 21,
#     rf_annual: Optional[float] = None
# ) -> pd.DataFrame:
#     """
#     Costruisce fattori MKT, SMB, MOM (+ RF opzionale)
#     usando esclusivamente stocks_data e company_data.
#     """

#     prices = stocks_data.copy().sort_index()

#     # --- RETURNS ---
#     if use_logrets:
#         rets = np.log(prices).diff().dropna()
#     else:
#         rets = prices.pct_change().dropna()

#     rets = rets.replace([np.inf, -np.inf], np.nan).fillna(0)

#     # =========================================================
#     # 1️⃣ MKT (market-cap weighted universe return)
#     # =========================================================
#     if "marketCap" not in company_data.columns:
#         raise ValueError("company_data must contain 'marketCap' column")

#     mc = company_data["marketCap"].reindex(prices.columns).astype(float)
#     mc = mc.clip(lower=0)

#     if mc.sum() <= 0:
#         weights = pd.Series(1 / len(mc), index=mc.index)
#     else:
#         weights = mc / mc.sum()

#     MKT = (rets * weights.values).sum(axis=1)
#     MKT.name = "MKT"

#     # =========================================================
#     # 2️⃣ SMB (Small minus Big)
#     # =========================================================
#     mc_rank = mc.dropna().sort_values()
#     n = len(mc_rank)
#     bucket = max(1, int(np.ceil(smb_q * n)))

#     small = mc_rank.index[:bucket]
#     big   = mc_rank.index[-bucket:]

#     SMB = rets[small].mean(axis=1) - rets[big].mean(axis=1)
#     SMB.name = "SMB"

#     # =========================================================
#     # 3️⃣ MOM (12-1 Momentum)
#     # =========================================================
#     prices_ffill = prices.ffill()

#     p_start = prices_ffill.shift(mom_skip_days + mom_formation_days)
#     p_end   = prices_ffill.shift(mom_skip_days)

#     if use_logrets:
#         mom_raw = np.log(p_end) - np.log(p_start)
#     else:
#         mom_raw = (p_end / p_start) - 1

#     MOM = mom_raw.mean(axis=1)
#     MOM = MOM.reindex(rets.index).fillna(0)
#     MOM.name = "MOM"

#     # =========================================================
#     # Assemble factors
#     # =========================================================
#     factors = pd.concat([MKT, SMB, MOM], axis=1)

#     # RF opzionale (log daily)
#     if rf_annual is not None:
#         rf_daily = np.log1p(rf_annual) / 252
#         factors["RF"] = rf_daily

#     return factors.fillna(0)

In [ ]:
# # stocks_data, company_data = fetch_data_and_companies(
# #     tickers,
# #     download_start_date,
# #     end_date,
# #     normalize=False,
# #     auto_adjust=False
# # )

# factors = build_europe_factors_from_internal_data(
#     stocks_data,
#     company_data,
#     rf_annual=0.01
# )

# res = rolling_factor_regression_from_pf(
#     pf_rot_std,
#     factors[["MKT", "SMB", "MOM"]],
#     window=252
# )

# # res["summary"]

# # res = rolling_factor_regression_from_pf(
# #     pf_rot,
# #     factors,
# #     window=252,
# # )

# res["summary"]

In [ ]:
# nav = pf_rot_std.value()
# if isinstance(nav, pd.DataFrame):
#     nav = nav.iloc[:,0]

# asset_rets = np.log(nav).diff().dropna()
# print("Asset returns start:", asset_rets.index.min())
# print("Asset returns end:", asset_rets.index.max())
# print("Asset len:", len(asset_rets))

# print("Factors start:", factors.index.min())
# print("Factors end:", factors.index.max())
# print("Factors len:", len(factors))

# common_index = asset_rets.index.intersection(factors.index)
# print("Common len:", len(common_index))

In [ ]:
# print(factors.isna().sum())
# print(factors.head(300))


In [ ]:
# nav = pf_rot_std.value()
# if isinstance(nav, pd.DataFrame):
#     nav = nav.iloc[:, 0]

# asset_rets = np.log(nav).diff().dropna()

# common_index = asset_rets.index.intersection(factors.index)
# y = asset_rets.reindex(common_index)

# print("Varianza y:", y.var())
# print("Min return:", y.min())
# print("Max return:", y.max())
# print("Percentuale zero returns:", (y == 0).mean())

In [ ]:
# import statsmodels.api as sm

# nav = pf_rot_std.value()
# if isinstance(nav, pd.DataFrame):
#     nav = nav.iloc[:,0]

# asset_rets = np.log(nav).diff().dropna()
# common_index = asset_rets.index.intersection(factors.index)

# y = asset_rets.reindex(common_index).iloc[-252:]
# X = factors[["MKT","SMB","MOM"]].reindex(common_index).iloc[-252:]

# Xc = sm.add_constant(X)
# model = sm.OLS(y, Xc).fit()

# print(model.summary())

## Send Stats

In [ ]:
# Report di performance e statistiche
# %run u_functions.ipynb

# Send Performance Report
sender_email = "lf27963@gmail.com"
sender_password = 'vlnuulfibbokalye'
# recipient_email='lf27963@gmail.com,customercare.ec@gmail.com' 
# recipient_email='customercare.ec@gmail.com' 
recipient_email = 'lf27963@gmail.com' 
# recipient_email = ""

# 
show_report=True
auto_adjust=True

out_p = run_rotational_portfolio_performance(
    portfolio=portfolio,
    analisys_start_date=analisys_start_date,
    analisys_end_date=analisys_end_date,
    wfo_results_dir=wfo_results_dir,
    sender_email=sender_email,
    sender_password=sender_password,
    recipient_email=recipient_email,
    show_report=show_report,
    debug=debug,
    verbose=verbose,
    auto_adjust=auto_adjust
)

## Send Rebalance Report (Run)

In [ ]:
# %run _bootstrap_dev.ipynb

sender_email = "lf27963@gmail.com"
sender_password = 'vlnuulfibbokalye'
recipient_email='lf27963@gmail.com,customercare.ec@gmail.com' 
# recipient_email='customercare.ec@gmail.com' 
recipient_email = 'lf27963@gmail.com' 
# recipient_email = ""
subject = None 
verbose = True
debug = False
dry_run = False

# Report date ....
report_end_date = None

out_r = r_run_portfolio(
    portfolio=portfolio,
    report_end_date=report_end_date,     
    year=year,
    wfo_results_dir=wfo_results_dir,
    sender_email=sender_email,
    sender_password=sender_password,
    recipient_email=recipient_email,
    subject=subject,
    verbose=verbose,
    dry_run=dry_run,
    debug=debug
)

## Engine Sanity Check

In [ ]:
# %run _bootstrap_dev.ipynb
print("CALL prices id:", id(stocks_data))
health_df, ticker_df, selection_log, details = build_engine_health_check(
    pf_rot_std,
    sel_tickers,
    prices=stocks_data,             # se ce l’hai: molto meglio
    start_date=start_date,
    end_date=end_date,
    include_prev=True
)

my_display(health_df, "Health check motore rotazionale")
my_display(ticker_df, "Per-ticker diagnostics (selezioni + held-return)")
my_display(selection_log, "Audit selezioni (data -> ticker)")
my_display(sel_tickers)

## Monte Carlo per Portfolios Rotazionali - Guida Rapida

### Panoramica

Hai **3 strumenti MC complementari** per validare strategie rotazionali:

```
1. Block Bootstrap (returns robustness)
2A. Ranking Noise - Single Strategy (selection robustness)  
2B. Ranking Noise - WFO Multi-Window (parameter stability over time)
```

---

### 1️⃣ BLOCK BOOTSTRAP (Approccio A)

#### 📝 Cosa Fa

**Randomizzazione:**
- Resample **blocchi contigui** di returns giornalieri (es. 20 giorni)
- Preserva autocorrelazione temporale (momentum è autocorrelato!)
- Usa le **STESSE selezioni** del portfolio reale
- Simula 10,000 scenari "what-if returns fossero leggermente diversi"

**Esempio pratico:**
```
Portfolio reale:
  2020-01-15: seleziona [AAPL, MSFT, GOOGL]
  Returns effettivi: AAPL +2%, MSFT -1%, GOOGL +3%

MC Simulazione #5473:
  2020-01-15: stessa selezione [AAPL, MSFT, GOOGL]  ← NON cambia
  Returns simulati: AAPL +1.8%, MSFT -0.5%, GOOGL +2.7%  ← cambia (bootstrap)
```

#### 🎯 A Cosa Serve

Risponde: **"La performance è robusta o dipende da returns specifici?"**

- Testa sensibilità a **variazioni micro nei returns**
- Genera **confidence intervals** (es. "con 90% prob, return sarà tra +45% e +118%")
- Identifica **tail risk** (worst 5% scenarios)

#### ✅ Cosa Deduci

**Se actual performance è:**

| Percentile | Interpretazione | Azione |
|------------|-----------------|--------|
| 25-75% | ✅ **ROBUSTO** - Performance normale, non lucky | Deploy con confidence |
| 75-85% | 🟡 **Above average** - Buono ma monitora | Deploy, aspettati mean reversion |
| > 85% | 🟠 **High** - Possibile overfitting | Cautela, riduci position sizing |
| > 95% | 🔴 **Extreme tail** - Lucky o overfit grave | NON deploy, rivedere parametri |

**Metriche chiave:**
- **Final Return CI**: "Aspettati tra +50% e +120% con 90% prob"
- **Max DD worst 5%**: "Nel 5% worst-case, DD può arrivare a -38%"
- **Sharpe distribution**: Variabilità risk-adjusted performance

#### 💡 Esempio Decisione

```
Block Bootstrap Results:
  Actual Return: +85%
  MC Median: +78%
  Percentile: 65%  ← NORMALE
  
  Actual DD: -18%
  MC Worst 5%: -35%  ← Sei stato fortunato sui DD
  
→ DECISIONE: Deploy ma preparati a DD fino a -35% in scenari sfortunati
```

---

### 2️⃣ RANKING NOISE (Approccio B)

#### 📝 Cosa Fa

**Randomizzazione:**
- Aggiunge **noise gaussiano** al ranking momentum/risk-parity
- Titolo al 50° percentile → può diventare 45°-55° con noise 5%
- Le **selezioni cambiano** a ogni simulazione
- Simula 1,000 scenari "what-if ranking fosse leggermente impreciso"

**Esempio pratico:**
```
Ranking reale (momentum):
  1. AAPL (rank=0.95)
  2. MSFT (rank=0.89)
  3. GOOGL (rank=0.82)
  ...
  Top 3 selezionati: [AAPL, MSFT, GOOGL]

MC Simulazione #342 (noise 5%):
  1. MSFT (rank=0.91)  ← era 2°, ora 1°
  2. AAPL (rank=0.90)  ← era 1°, ora 2°  
  3. NVDA (rank=0.84)  ← era 4°, ora 3° (GOOGL out!)
  Top 3 selezionati: [MSFT, AAPL, NVDA]  ← 33% cambio!
```

#### 🎯 A Cosa Serve

Risponde: **"Le selezioni sono robuste o sensibili a noise micro nel ranking?"**

- Testa **stabilità selezioni** (overlap baseline vs noise)
- Misura **fragility parametri** (sharpe degradation con noise)
- Identifica se strategia dipende da **ranking perfetto** (irrealistico)

#### ✅ Cosa Deduci

**Selection Overlap (metrica principale):**

| Overlap | Rating | Interpretazione | Azione |
|---------|--------|-----------------|--------|
| > 80% | 🟢 **EXCELLENT** | Selezioni molto stabili | Deploy con confidence |
| 65-80% | ✅ **GOOD** | Ragionevolmente stabili | Deploy, è normale |
| 50-65% | 🟡 **MODERATE** | Moderatamente fragili | Cautela, considera lookback più lunghi |
| < 50% | 🔴 **FRAGILE** | Troppo sensibili a noise | NON deploy, parametri troppo aggressivi |

**Sharpe Degradation:**

| Degradation | Rating | Interpretazione |
|-------------|--------|-----------------|
| < 5% | 🟢 **EXCELLENT** | Performance robusta |
| 5-15% | ✅ **GOOD** | Accettabile |
| 15-30% | 🟡 **MODERATE** | Sensibile |
| > 30% | 🔴 **WEAK** | Overfit probabile |

#### 💡 Esempio Decisione

```
Ranking Noise Results:
  Avg Overlap: 58%  ← FRAGILE (< 60%)
  Sharpe Degradation: -22%  ← SENSIBILE
  
→ DIAGNOSI: Parametri troppo aggressivi
→ AZIONE: 
  - Aumenta momentum_lookback (20→60 giorni)
  - Aumenta n_top (3→8 ticker)
  - Rimuovi filtri extra (filter_volatility=False)
```

---

### 🔀 2A vs 2B: Single Strategy vs WFO Multi-Window

#### **2A. RANKING NOISE - Single Strategy**

**Usa quando:**
- Hai parametri fissi (non da WFO)
- Vuoi testare 1 set di parametri su tutto storico
- Quick validation pre-deployment

**Esempio:**
```python
# Parametri fissi
params = {'n_top': 5, 'momentum_lookback': 60, ...}

# Test su tutto storico 2020-2025
mc = monte_carlo_ranking_noise(
    stocks_data,  # 2020-2025 completo
    benchmark_data,
    params=params,
    n_simulations=1_000
)

# Output: robustezza parametri su INTERO periodo
```

**Output:** 1 rating complessivo (ROBUST/ACCEPTABLE/RISKY)

---

#### **2B. RANKING NOISE - WFO Multi-Window**

**Usa quando:**
- Hai WFO con parametri diversi per anno
- Vuoi validare PROCESSO WFO completo
- Identificare quali anni hanno parametri fragili

**Esempio:**
```python
# WFO ha parametri per ogni anno
wfo_summary:
  2020: {n_top: 3, lookback: 20}
  2021: {n_top: 5, lookback: 60}  
  2022: {n_top: 8, lookback: 120}
  ...

# Test OGNI finestra separatamente
wfo_mc = monte_carlo_wfo_per_window(
    stocks_data,
    benchmark_data,
    wfo_summary=wfo_testable,  # 6 finestre
    n_simulations=1_000
)

# Output: 6 rating separati (uno per anno)
```

**Output:** Rating **per finestra** + trend temporale

**Cosa deduci in più:**

| Pattern | Interpretazione | Azione |
|---------|-----------------|--------|
| Maggioranza ROBUST | ✅ WFO efficace | Deploy con confidence |
| Mix ROBUST/RISKY | 🟡 Regime-dependent | Analizza pattern (bear years fragili?) |
| Trend negativo recente | 🔴 Parametri si degradano | NON deploy, strategia aging |
| Solo 2022 RISKY | 🟢 Normale (bear market) | Ok se altri anni robusti |

**Esempio diagnostica avanzata:**
```
WFO MC Results:
  2020: ROBUST (overlap 82%)
  2021: ROBUST (overlap 79%)
  2022: RISKY (overlap 52%)  ← Bear market
  2023: ROBUST (overlap 81%)
  2024: ACCEPTABLE (overlap 68%)
  2025: ACCEPTABLE (overlap 71%)
  
→ PATTERN: Solo 2022 fragile (anno bear)
→ DECISIONE: Normale, bear markets creano instabilità
→ AZIONE: Deploy, ma preparati a review se 2026 è bear
```

---

### 🎯 Quale Usare Quando?

#### **Workflow Validation Completo**

```
┌─────────────────────────────────────┐
│  Walk-Forward Optimization (WFO)   │
│  → Best parameters per anno         │
└──────────┬──────────────────────────┘
           │
    ┌──────┴───────┐
    │              │
    ▼              ▼
┌─────────┐   ┌─────────────────┐
│ METODO  │   │ METODO          │
│ 2B      │   │ 1 + 2A          │
│ WFO MC  │   │ (combinato)     │
└─────────┘   └─────────────────┘
    │              │
    ▼              ▼
Per-window      Anno prossimo
analysis        validation
(storico)       (deploy)
```

#### **Scenario 1: Validation Storica Completa**

```python
# Usa 2B per analizzare tutto lo storico WFO
wfo_mc = monte_carlo_wfo_per_window(...)

# Risultato: sai quali anni erano robusti/fragili
# Utile per: capire pattern, identificare regime-dependency
```

#### **Scenario 2: Pre-Deployment Anno Prossimo**

```python
# Step 1: Block Bootstrap (Approccio 1)
pf_2026 = build_rotational_portfolios_vbt(..., **params_2026)
mc_bootstrap = monte_carlo_block_bootstrap_rotational(pf_2026, ...)

# Step 2: Ranking Noise Single (Approccio 2A)  
mc_noise = monte_carlo_ranking_noise(..., params=params_2026)

# Decisione combinata:
if (bootstrap_percentile < 0.85) and (noise_overlap > 0.65):
    print("✅ DEPLOY APPROVED")
```

---

### 📊 Decision Matrix Finale

#### **Combina TUTTI i risultati**

| Block Bootstrap | Ranking Noise | WFO Trend | Decisione |
|-----------------|---------------|-----------|-----------|
| Percentile 25-75% | Overlap > 65% | Recent ROBUST | ✅ **DEPLOY** full size |
| Percentile 75-85% | Overlap > 65% | Recent ROBUST | ✅ **DEPLOY** 70% size |
| Percentile > 85% | Overlap > 65% | - | 🟡 **DEPLOY** 50% size, monitor |
| Percentile < 75% | Overlap < 60% | - | 🔴 **DON'T DEPLOY** |
| Any | Overlap < 50% | Recent RISKY | 🔴 **DON'T DEPLOY** |

### **Red Flags Critici** (non deployare)

1. 🚩 Block Bootstrap percentile > 95% **AND** Ranking overlap < 60%
   - Troppo fortunato + selezioni fragili = overfitting grave

2. 🚩 WFO trend negativo (ultime 3 finestre peggiorano)
   - Strategia si sta degradando nel tempo

3. 🚩 Sharpe degradation > 30% con noise 5%
   - Performance collassa con minima perturbazione

---

## 🎓 Summary One-Liner

**Block Bootstrap:** _"Se returns fossero diversi, performance regge?"_  
**Ranking Noise (2A):** _"Se ranking fosse impreciso, selezioni cambiano?"_  
**WFO Multi-Window (2B):** _"Parametri erano robusti anno su anno?"_

**Best Practice:** Usa **TUTTI E TRE** prima di ogni deployment!

In [ ]:
ci_results, ci_summary_df, skill_results, skill_summary_df = run_all_mc_methods_rotational(
    pf_rot              = pf_rot_std,
    pf_rot_base         = pf_rot_std_base,
    regime              = regime,
    sel_tickers         = sel_tickers_std,
    sel_tickers_base    = sel_tickers_std_base,
    stocks_data         = stocks_data,
    benchmark_data      = benchmark_data,
    tickers_master      = tickers,
    init_cash           = init_cash,
    n_simulations       = 100,
    seed                = 42,
    block_size          = 10,
    vol_window          = 60,
    n_vol_quantiles     = 3,
    show_method_plots   = False,           # plot off al primo giro
    show_method_summaries = True,
)

ci_summary_df
skill_summary_df

In [ ]:
ci_results, ci_summary_df, skill_results, skill_summary_df = run_all_mc_methods_rotational(
    pf_rot              = pf_rot_std,
    pf_rot_base         = pf_rot_std_base,
    regime              = regime,
    sel_tickers         = sel_tickers_std,
    sel_tickers_base    = sel_tickers_std_base,
    stocks_data         = stocks_data,
    benchmark_data      = benchmark_data,
    tickers_master      = tickers,
    init_cash           = init_cash,
    n_simulations       = 1000,
    seed                = 42,
    block_size          = 10,
    vol_window          = 60,
    n_vol_quantiles     = 3,
    show_method_plots   = True,
    show_method_summaries = True,
)

In [ ]:
display(ci_summary_df)
display(skill_summary_df)

In [ ]:
# # 1. Quale pf_rot stai passando?
# print("pf_rot stats (Risk ON/OFF):")
# print(f"  CAGR:   {pf_rot_std.annualized_return():.4f}")
# print(f"  MaxDD:  {pf_rot_std.max_drawdown():.4f}")
# print(f"  Sharpe: {pf_rot_std.sharpe_ratio():.4f}")

# print("\npf_rot_base stats (NO Risk ON/OFF):")
# print(f"  CAGR:   {pf_rot_std_base.annualized_return():.4f}")
# print(f"  MaxDD:  {pf_rot_std_base.max_drawdown():.4f}")
# print(f"  Sharpe: {pf_rot_std_base.sharpe_ratio():.4f}")

In [ ]:
# Dopo aver lanciato la run, ispeziona ci_results e skill_results

# Blocco A
print("=== Blocco A — actual_metrics (interni) ===")
print("A1 IID:")
print(ci_results['iid_bootstrap']['actual_metrics'])
print("\nA2 Block:")
print(ci_results['block_bootstrap']['actual_metrics'])

# Blocco B
print("\n=== Blocco B — actual_metrics (interni) ===")
print("B1 Reshuffle:")
print(skill_results['rotation_reshuffle']['actual_metrics'])
print("\nB2 Timing:")
print(skill_results['rebalance_timing']['actual_metrics'])

# E l'equity sorgente
import pandas as pd
print("\n=== Equity sources ===")
eq_pf_rot = pf_rot_std.value()
eq_pf_rot_base = pf_rot_std_base.value()
print(f"pf_rot equity: start={eq_pf_rot.iloc[0]:.0f}, end={eq_pf_rot.iloc[-1]:.0f}, days={len(eq_pf_rot)}")
print(f"pf_rot_base equity: start={eq_pf_rot_base.iloc[0]:.0f}, end={eq_pf_rot_base.iloc[-1]:.0f}, days={len(eq_pf_rot_base)}")

# Manual CAGR check con la stessa formula del modulo MC
def manual_cagr(equity, trading_days=252):
    n = len(equity)
    years = n / trading_days
    return (equity.iloc[-1] / equity.iloc[0]) ** (1/years) - 1

print(f"\nManual CAGR pf_rot:      {manual_cagr(eq_pf_rot):.4f}")
print(f"Manual CAGR pf_rot_base: {manual_cagr(eq_pf_rot_base):.4f}")

---

## Monte Carlo Validation — Nuova API (Post-WFO)

Due wrapper separati e non mescolabili:
- **`run_mc_confidence_intervals_rotational`** → Blocco A: intervalli di confidenza (quantili p5/p50/p95)
- **`run_mc_skill_tests_rotational`** → Blocco B: skill tests (p-value, permutation tests)

**Non mescolare** `ci_summary_df` (quantili) con `skill_summary_df` (p-value).

Prerequisito: `results = run_wfo_pipeline(...)` già eseguito nella sessione.


In [ ]:
# ── Estrai variabili dal risultato della pipeline ─────────────────────────────
pf_rot_std           = results_std['pf_rot']           # con Risk ON/OFF
pf_rot_std_base      = results_std['pf_rot_base']      # senza Risk ON/OFF
regime               = results_std['regime']           # pd.Series 0/1 (None se non-clustered)
sel_tickers_std      = results_std['sel_tickers']      # con Risk ON/OFF (ha colonna 'universe')
sel_tickers_std_base = results_std['sel_tickers_base'] # senza Risk ON/OFF

# benchmark_data, stocks_data, tickers, init_cash: variabili già presenti in sessione

# =============================================================================
# BLOCCO A — Confidence Intervals (intervalli di confidenza sulle metriche)
# =============================================================================
# Risponde a: "Il risultato OOS è robusto o frutto di una sequenza fortunata?"
# Produce: quantili p5/p25/p50/p75/p95  — NON p-value

ci_results, ci_summary_df = run_mc_confidence_intervals_rotational(
    pf_rot          = pf_rot_std,
    pf_rot_base     = pf_rot_std_base,
    regime          = regime,           # None → A3 skippato con warning
    benchmark_data  = benchmark_data,
    init_cash       = init_cash,
    n_simulations   = 1000,
    seed            = 42,
    block_size      = 10,
    show_method_plots     = True,
    show_method_summaries = True,
)

# ci_results keys: 'iid_bootstrap', 'block_bootstrap', 'regime_block' (None se skippato)
# Ogni entry ha: equity_curves, metrics_per_sim, percentiles, actual_metrics, actual_quantile_position

# Accesso diretto:
ci_block = ci_results['block_bootstrap']
print("CAGR p5/p50/p95:", ci_block['percentiles']['p5']['CAGR'],
      ci_block['percentiles']['p50']['CAGR'],
      ci_block['percentiles']['p95']['CAGR'])
print("CAGR actual:", ci_block['actual_metrics']['CAGR'])
print("CAGR quantile position:", f"{ci_block['actual_quantile_position']['CAGR']:.0%}")

# Tabella riepilogativa (contiene SOLO quantili — niente p-value)
display(ci_summary_df)

# =============================================================================
# BLOCCO B — Skill Tests (p-value — NON intervalli di confidenza)
# =============================================================================
# Risponde a: "La performance viene dalla skill o dall'esposizione all'universo?"
# Produce: p-value per ogni metrica — NON quantili
#
# Raccomandazione: usare pf_rot_base + stocks_data puro (no risk_off_data)
# per evitare mismatch tra ticker in sel_tickers_base e stocks_data.

skill_results, skill_summary_df = run_mc_skill_tests_rotational(
    pf_rot          = pf_rot_std_base,       # portafoglio base (no risk on/off)
    sel_tickers     = sel_tickers_std_base,  # selezioni base
    stocks_data     = stocks_data,           # prezzi puri (no risk_off_data)
    benchmark_data  = benchmark_data,
    regime          = regime,
    tickers_master  = tickers,              # fall-back universe per B1
    init_cash       = init_cash,
    n_simulations   = 1000,
    seed            = 42,
    vol_window      = 60,
    n_vol_quantiles = 3,
    show_method_plots     = True,
    show_method_summaries = True,
)

# skill_results keys: 'rotation_reshuffle' (B1), 'rebalance_timing' (B2)
# Ogni entry ha: equity_curves, metrics_per_sim, actual_metrics, p_values, interpretation

print()
print("B1 interpretation:", skill_results['rotation_reshuffle']['interpretation'])
print("B2 interpretation:", skill_results['rebalance_timing']['interpretation'])

# Tabella riepilogativa (contiene SOLO p-value — niente quantili)
display(skill_summary_df)

# # =============================================================================
# # ALTERNATIVA — Tutto in una chiamata (usa pf_rot per A, pf_rot_base per B)
# # =============================================================================
# ci_results, ci_summary_df, skill_results, skill_summary_df = run_all_mc_methods_rotational(
#     pf_rot              = pf_rot_std,
#     pf_rot_base         = pf_rot_std_base,
#     regime              = regime,
#     sel_tickers         = sel_tickers_std,
#     sel_tickers_base    = sel_tickers_std_base,
#     stocks_data         = stocks_data,
#     benchmark_data      = benchmark_data,
#     tickers_master      = tickers,
#     init_cash           = init_cash,
#     n_simulations       = 1000,
#     seed                = 42,
#     block_size          = 10,
#     vol_window          = 60,
#     n_vol_quantiles     = 3,
#     show_method_plots   = True,
#     show_method_summaries = True,
# )
# # Tabella riepilogativa (contiene SOLO quantili — niente p-value)
# display(ci_summary_df)

# # Tabella riepilogativa (contiene SOLO p-value — niente quantili)
# display(skill_summary_df)


In [ ]:
# report_text = generate_mc_validation_report(
#     ci_results       = ci_results,
#     ci_summary_df    = ci_summary_df,
#     skill_results    = skill_results,
#     skill_summary_df = skill_summary_df,
#     pf_rot           = pf_rot_std,
#     pf_rot_base      = pf_rot_std_base,
#     sel_tickers      = sel_tickers_std,
#     benchmark_data   = benchmark_data,
#     portfolio_name   = "R_Asset_Std_2026-04",   # o il nome che preferisci
#     mc_setup         = {
#         "n_simulations": 1000,
#         "seed": 42,
#         "block_size": 10,
#         "vol_window": 60,
#         "n_vol_quantiles": 3,
#     },
#     save_path        = None,   # primo giro: solo a video, no salvataggio
# )

# print(report_text)

report_text = generate_mc_validation_report(
    ci_results       = ci_results,
    ci_summary_df    = ci_summary_df,
    skill_results    = skill_results,
    skill_summary_df = skill_summary_df,
    pf_rot           = pf_rot_std,
    pf_rot_base      = pf_rot_std_base,
    sel_tickers      = sel_tickers_std,
    benchmark_data   = benchmark_data,
    portfolio_name   = "R_Asset_Std_2026-04",
    mc_setup         = {"n_simulations": 1000, "seed": 42, "block_size": 10,
                        "vol_window": 60, "n_vol_quantiles": 3},
    save_path        = 'reports/MC_validation/',
    overwrite        = True,
)


In [ ]:
# Ispeziona il distribution_shape calcolato sui dati reali
print("=== B1 distribution_shape ===")
for metric, shape in skill_results['rotation_reshuffle']['distribution_shape'].items():
    print(f"\n{metric}:")
    print(f"  is_unimodal:   {shape['is_unimodal']}")
    print(f"  shape_warning: {shape['shape_warning']}")
    print(f"  kurtosis:      {shape['kurtosis']:+.3f}")
    print(f"  skewness:      {shape['skewness']:+.3f}")

print("\n=== B2 distribution_shape ===")
for metric, shape in skill_results['rebalance_timing']['distribution_shape'].items():
    print(f"\n{metric}:")
    print(f"  is_unimodal:   {shape['is_unimodal']}")
    print(f"  shape_warning: {shape['shape_warning']}")
    print(f"  kurtosis:      {shape['kurtosis']:+.3f}")
    print(f"  skewness:      {shape['skewness']:+.3f}")

In [ ]:
# =============================================================================
# STABILITY ANALYSIS — smoke test
# =============================================================================
# best_params_row deve essere definito a monte (output WFO)
# Esempio: best_params_row = summary_df.iloc[0]

periods = _split_history_into_periods("2015-01-01", "2024-12-31", k=3)

for s, e in periods:
    val = _evaluate_ptf_on_period(
        {"stocks_data": stocks_data, "init_cash": 100_000},
        EngineParams.from_dict(best_params_row),
        s, e,
        metric="CAGR",
    )
    print(f"{s.date()} → {e.date()}: CAGR={val:.2%}")

In [ ]:
# =============================================================================
# STABILITY ANALYSIS — Step 1.2 smoke test: _evaluate_flag_stability
# =============================================================================
# Cosa guardare nell'output:
#   - coherent_sign=True: il flag ha effetto stabile e uniforme su tutti i
#     sotto-periodi → recommended_value è affidabile.
#   - coherent_sign=False: l'effetto cambia segno tra periodi (instabile) o
#     mancano dati. Guarda delta_per_period_per_anchor per capire dove e
#     quanto varia il delta per ciascun valore di n_top.
#   - mean_delta positivo: in media flag=True batte flag=False su questa metrica.
#   - mean_delta vicino a 0: il flag non ha impatto apprezzabile.

import pprint

# base_params dalla stessa riga del WFO summary usata in Step 1.1
wfo_df = pd.read_csv('../../inputs/WFO_R_RUN_RESULTS/Alpha Euro_2026.wfo_summary.csv')
base_params = dict(wfo_df.iloc[0])

print("=== base_params (da wfo_summary row 0) ===")
for k_name, v in base_params.items():
    print(f"  {k_name}: {v}")
print()

result = _evaluate_flag_stability(
    ptf_config      = {"stocks_data": stocks_data, "init_cash": 100_000},
    base_params     = base_params,
    flag_name       = "filter_ema",
    full_start_date = "2015-01-01",
    full_end_date   = "2024-12-31",
    metric          = "CAGR",
    k               = 3,
    n_top_anchors   = None,   # default: [3, 5, 8]
)

print("=== _evaluate_flag_stability(filter_ema) ===")
for key, val in result.items():
    if key == "delta_per_period_per_anchor":
        print(f"  {key}:")
        periods = _split_history_into_periods("2015-01-01", "2024-12-31", k=3)
        for i, (row, (s, e)) in enumerate(zip(val, periods)):
            formatted = [f"{d:+.4f}" if not __import__('math').isnan(d) else "NaN" for d in row]
            print(f"    period {i+1} [{s.date()}→{e.date()}]: {formatted}")
    elif key == "delta_per_period":
        formatted = [f"{d:+.4f}" if not __import__('math').isnan(d) else "NaN" for d in val]
        print(f"  {key}: {formatted}")
    elif isinstance(val, float):
        print(f"  {key}: {val:+.4f}" if not __import__('math').isnan(val) else f"  {key}: NaN")
    else:
        print(f"  {key}: {val}")

In [ ]:
# =============================================================================
# STABILITY ANALYSIS — Step 1.2 extended smoke test: full Alpha Euro universe
# =============================================================================
# Razionale: il test ridotto (5 ticker) produce delta identici tra anchor perché
# n_top >= universe_size → selezione completa, nessuna variazione per anchor.
# Con 35 ticker gli anchor [3, 5, 8] producono selezioni davvero diverse.
#
# Cosa guardare:
#   - delta_per_period_per_anchor: se i valori variano tra anchor nello stesso
#     periodo, l'interazione flag × concentrazione è reale.
#   - coherent_sign=True: su universo adeguato il segnale è più affidabile.
#   - Confronto con il test ridotto: stessa recommendation ma con evidenza vera.

import yfinance as yf, math
_eu_tickers = ['ENEL.MI', 'ISP.MI', 'UCG.MI', 'ENI.MI', 'STLAM.MI', 'PRY.MI', 'G.MI', 'LDO.MI', 'PST.MI', 'SAP.DE', 'SIE.DE', 'IFX.DE', 'DTE.DE', 'BAYN.DE', 'ALV.DE', 'MRK.DE', 'BAS.DE', 'RWE.DE', 'MC.PA', 'OR.PA', 'AIR.PA', 'SAN.PA', 'BNP.PA', 'AI.PA', 'KER.PA', 'HO.PA', 'ML.PA', 'ITX.MC', 'IBE.MC', 'SAN.MC', 'REP.MC', 'TEF.MC', 'ACS.MC', 'ACX.MC', 'CLNX.MC', 'GRF.MC']
_eu_data = yf.download(_eu_tickers, start='2013-01-01', end='2025-01-01',
                        auto_adjust=True, progress=False)['Close']
# drop tickers with < 80% coverage
_min_rows = int(len(_eu_data) * 0.80)
_eu_data = _eu_data.dropna(thresh=_min_rows, axis=1).dropna(how='all').ffill()
print(f"Alpha Euro tickers available: {_eu_data.shape[1]} / {len(_eu_tickers)}")
print(f"Tickers: {list(_eu_data.columns)}")
print()

wfo_df = pd.read_csv('../../inputs/WFO_R_RUN_RESULTS/Alpha Euro_2026.wfo_summary.csv')
base_params = dict(wfo_df.iloc[0])

result_full = _evaluate_flag_stability(
    ptf_config      = {"stocks_data": _eu_data, "init_cash": 100_000},
    base_params     = base_params,
    flag_name       = "filter_ema",
    full_start_date = "2015-01-01",
    full_end_date   = "2024-12-31",
    metric          = "CAGR",
    k               = 3,
    n_top_anchors   = None,
)

periods = _split_history_into_periods("2015-01-01", "2024-12-31", k=3)
print("=== _evaluate_flag_stability(filter_ema) — Alpha Euro FULL ===")
for key, val in result_full.items():
    if key == "delta_per_period_per_anchor":
        print(f"  {key}:")
        for i, (row, (s, e)) in enumerate(zip(val, periods)):
            fmt = [f"{d:+.4f}" if not math.isnan(d) else "NaN" for d in row]
            print(f"    period {i+1} [{s.date()}→{e.date()}]: {fmt}")
    elif key == "delta_per_period":
        fmt = [f"{d:+.4f}" if not math.isnan(d) else "NaN" for d in val]
        print(f"  {key}: {fmt}")
    elif isinstance(val, float):
        print(f"  {key}: {val:+.4f}" if not math.isnan(val) else f"  {key}: NaN")
    else:
        print(f"  {key}: {val}")

# ── Anchor comparison vs reduced test ────────────────────────────────────────
print()
print("=== Anchor comparison: 5-ticker vs full universe ===")
prev_anchor = [[+0.0005, +0.0005, +0.0005],
               [-0.0201, -0.0201, -0.0201],
               [-0.0046, -0.0046, -0.0046]]
anchors = result_full["n_top_anchors"]
for i, (s, e) in enumerate(periods):
    prev = prev_anchor[i]
    full = result_full["delta_per_period_per_anchor"][i]
    print(f"  Period {i+1} [{s.date()}→{e.date()}]:")
    for j, anc in enumerate(anchors):
        d_prev = prev[j]; d_full = full[j]
        diff = d_full - d_prev if not math.isnan(d_full) else float("nan")
        flag = "DIVERSE" if abs(diff) > 1e-6 else "identical"
        print(f"    n_top={anc}: 5-ticker={d_prev:+.4f}  full={d_full:+.4f}  diff={diff:+.4f}  [{flag}]")
print()
print(f"  5-ticker: incoherent, mean_delta=-0.0081, recommended_value=False")
print(f"  full    : {result_full['diagnostic_note']}, mean_delta={result_full['mean_delta']:+.4f}, recommended_value={result_full['recommended_value']}")

In [ ]:
# =============================================================================
# STABILITY ANALYSIS — Step 1.3 smoke test: reduce_grid_via_stability
# =============================================================================
# Tests full orchestrator on Alpha Euro (35 tickers) with param_grid_rotational_v3
# Expected: up to 4 flags evaluated, grid reduction from 3456 to <= 216 combinations.

import yfinance as yf, os, math
from datetime import datetime

# ── Load Alpha Euro full universe ────────────────────────────────────────────
_eu_tickers = ['ENEL.MI', 'ISP.MI', 'UCG.MI', 'ENI.MI', 'STLAM.MI', 'PRY.MI', 'G.MI', 'LDO.MI', 'PST.MI', 'SAP.DE', 'SIE.DE', 'IFX.DE', 'DTE.DE', 'BAYN.DE', 'ALV.DE', 'MRK.DE', 'BAS.DE', 'RWE.DE', 'MC.PA', 'OR.PA', 'AIR.PA', 'SAN.PA', 'BNP.PA', 'AI.PA', 'KER.PA', 'HO.PA', 'ML.PA', 'ITX.MC', 'IBE.MC', 'SAN.MC', 'REP.MC', 'TEF.MC', 'ACS.MC', 'ACX.MC', 'CLNX.MC', 'GRF.MC']
_eu_data = yf.download(_eu_tickers, start='2013-01-01', end='2025-01-01',
                        auto_adjust=True, progress=False)['Close']
_min_rows = int(len(_eu_data) * 0.80)
_eu_data = _eu_data.dropna(thresh=_min_rows, axis=1).dropna(how='all').ffill()
print(f'Alpha Euro tickers available: {_eu_data.shape[1]}')

# ── Define param_grid_rotational_v3 ─────────────────────────────────────────
param_grid_v3 = {
    'rebalance_frequency'      : ['ME', 'QE'],
    'momentum_lookback_days'   : [40, 60, 120, 180],
    'riskparity_lookback_days' : [40, 60, 120],
    'n_top'                    : [3, 5, 8],
    'use_acceleration'         : [True, False],
    'momentum_weight'          : [0.5, 0.7, 1.0],
    'filter_ema'               : [True, False],
    'filter_volatility'        : [True, False],
    'filter_min_momentum'      : [True, False],
}
from math import prod as _prod
print(f'param_grid_v3 original combinations: {_prod(len(v) for v in param_grid_v3.values())}')
print()

# ── Run reduce_grid_via_stability ────────────────────────────────────────────
reduced_grid, diag_report = reduce_grid_via_stability(
    ptf_config      = {'stocks_data': _eu_data, 'init_cash': 100_000},
    full_grid       = param_grid_v3,
    full_start_date = '2015-01-01',
    full_end_date   = '2024-12-31',
    metric          = 'CAGR',
    k               = 3,
    n_top_anchors   = None,
    verbose         = True,
)

# ── Print reduced_grid ───────────────────────────────────────────────────────
print()
print('=== reduced_grid ===')
for k_name, vals in reduced_grid.items():
    print(f'  {k_name}: {vals}')

# ── Full diagnostic report ───────────────────────────────────────────────────
print()
print('=== diagnostic_report (full, all columns) ===')
print(diag_report.to_string())

# ── Save CSV for historical audit ────────────────────────────────────────────
_report_dir = '../../data/stability_reports'
os.makedirs(_report_dir, exist_ok=True)
_today = datetime.now().strftime('%Y%m%d')
_report_path = f'{_report_dir}/alpha_euro_stability_{_today}.csv'
diag_report.to_csv(_report_path, index=False)
print(f'\nDiagnostic report saved to: {_report_path}')

In [ ]:
# =============================================================================
# STABILITY ANALYSIS — Step 1.4 smoke test: walk_forward_rotational
# auto_reduce_grid=False (Run A) vs auto_reduce_grid=True (Run B)
# =============================================================================
# Confronto su 3 anni OOS (2021-2023) per velocità del test.
# Run A: griglia completa 3456 combo; Run B: griglia ridotta via stability.

import yfinance as yf, time
from math import prod as _prod

# ── Load data ────────────────────────────────────────────────────────────────
_eu_tickers = ['ENEL.MI', 'ISP.MI', 'UCG.MI', 'ENI.MI', 'STLAM.MI', 'PRY.MI', 'G.MI', 'LDO.MI', 'PST.MI', 'SAP.DE', 'SIE.DE', 'IFX.DE', 'DTE.DE', 'BAYN.DE', 'ALV.DE', 'MRK.DE', 'BAS.DE', 'RWE.DE', 'MC.PA', 'OR.PA', 'AIR.PA', 'SAN.PA', 'BNP.PA', 'AI.PA', 'KER.PA', 'HO.PA', 'ML.PA', 'ITX.MC', 'IBE.MC', 'SAN.MC', 'REP.MC', 'TEF.MC', 'ACS.MC', 'ACX.MC', 'CLNX.MC', 'GRF.MC']
_eu_data = yf.download(_eu_tickers, start='2013-01-01', end='2024-12-31',
                        auto_adjust=True, progress=False)['Close']
_eu_data = _eu_data.dropna(thresh=int(len(_eu_data)*0.80), axis=1).dropna(how='all').ffill()
_bench = yf.download('^STOXX50E', start='2013-01-01', end='2024-12-31',
                      auto_adjust=True, progress=False)['Close'].squeeze()
print(f'Tickers: {_eu_data.shape[1]}, bench: {len(_bench)} rows')

# ── Compact grid (same structure as v3 but 2 lookbacks for speed) ────────────
param_grid_v3 = {
    'rebalance_frequency'      : ['ME', 'QE'],
    'momentum_lookback_days'   : [40, 60, 120, 180],
    'riskparity_lookback_days' : [40, 60, 120],
    'n_top'                    : [3, 5, 8],
    'use_acceleration'         : [True, False],
    'momentum_weight'          : [0.5, 0.7, 1.0],
    'filter_ema'               : [True, False],
    'filter_volatility'        : [True, False],
    'filter_min_momentum'      : [True, False],
}
print(f'Grid: {_prod(len(v) for v in param_grid_v3.values())} combinations')
print()

# ── Run A: baseline, no auto_reduce_grid ─────────────────────────────────────
print('=== Run A: auto_reduce_grid=False ===')
_t0 = time.time()
wfo_a = walk_forward_rotational(
    stocks_data=_eu_data, benchmark_data=_bench,
    param_grid=param_grid_v3,
    ratio='3:1', metric='Sharpe Ratio',
    start_date='2018-01-01', end_date='2024-01-01',
    auto_reduce_grid=False,
    verbose=True, plot=False, n_jobs=1,
)
_t_a = time.time() - _t0
print(f'Run A time: {_t_a:.1f}s')
print()

# ── Run B: auto_reduce_grid=True ─────────────────────────────────────────────
print('=== Run B: auto_reduce_grid=True ===')
_t0 = time.time()
wfo_b = walk_forward_rotational(
    stocks_data=_eu_data, benchmark_data=_bench,
    param_grid=param_grid_v3,
    ratio='3:1', metric='Sharpe Ratio',
    start_date='2018-01-01', end_date='2024-01-01',
    auto_reduce_grid=True,
    stability_metric='CAGR', stability_k=3,
    stability_report_dir='../../data/stability_reports',
    verbose=True, plot=False, n_jobs=1,
)
_t_b = time.time() - _t0
print(f'Run B time: {_t_b:.1f}s')
print()

# ── Comparison report ────────────────────────────────────────────────────────
print('=' * 60)
print('COMPARISON REPORT: Run A (full) vs Run B (auto_reduce_grid)')
print('=' * 60)
print(f'  Run A time : {_t_a:.1f}s')
print(f'  Run B time : {_t_b:.1f}s')
print(f'  Speedup    : {_t_a / _t_b:.1f}x')
print(f'  Run A windows: {len(wfo_a)}')
print(f'  Run B windows: {len(wfo_b)}')
print()
# Best params per window
_flag_cols = [c for c in wfo_a.columns if c in {'filter_ema','filter_volatility','filter_min_momentum','use_acceleration'}]
_num_cols  = [c for c in wfo_a.columns if c in {'momentum_lookback_days','riskparity_lookback_days','n_top','momentum_weight','rebalance_frequency'}]
print('Best params per window (flags only):')
print(f'  Run A:\n{wfo_a[_flag_cols].to_string()}')
print(f'  Run B:\n{wfo_b[_flag_cols].to_string()}')
print()
print('TestScore comparison:')
_cmp = wfo_a[['TestScore']].rename(columns={'TestScore':'TestScore_A'}).join(
    wfo_b[['TestScore']].rename(columns={'TestScore':'TestScore_B'}), how='outer'
)
_cmp['delta'] = _cmp['TestScore_B'] - _cmp['TestScore_A']
print(_cmp.to_string())
print(f'  Mean delta (B-A): {_cmp["delta"].mean():+.4f}')

In [ ]:
# =============================================================================
# DIAGNOSTICA: stability con metric=CAGR vs metric='Sharpe' — confronto flag
# Ipotesi: disallineamento metrica stability vs metrica WFO spiega la perdita
# di 0.52 Sharpe osservata nello smoke test Step 1.4.
# NOTA: cella non committata, in attesa di revisione risultati.
# =============================================================================

import yfinance as yf, math
from math import prod as _prod

# ── Load Alpha Euro full ─────────────────────────────────────────────────────
_eu_tickers = ['ENEL.MI', 'ISP.MI', 'UCG.MI', 'ENI.MI', 'STLAM.MI', 'PRY.MI', 'G.MI', 'LDO.MI', 'PST.MI', 'SAP.DE', 'SIE.DE', 'IFX.DE', 'DTE.DE', 'BAYN.DE', 'ALV.DE', 'MRK.DE', 'BAS.DE', 'RWE.DE', 'MC.PA', 'OR.PA', 'AIR.PA', 'SAN.PA', 'BNP.PA', 'AI.PA', 'KER.PA', 'HO.PA', 'ML.PA', 'ITX.MC', 'IBE.MC', 'SAN.MC', 'REP.MC', 'TEF.MC', 'ACS.MC', 'ACX.MC', 'CLNX.MC', 'GRF.MC']
_eu_data = yf.download(_eu_tickers, start='2013-01-01', end='2025-01-01',
    auto_adjust=True, progress=False)['Close']
_eu_data = _eu_data.dropna(thresh=int(len(_eu_data)*0.80), axis=1).dropna(how='all').ffill()
print(f'Tickers: {_eu_data.shape[1]}')

param_grid_v3 = {
    'rebalance_frequency'      : ['ME', 'QE'],
    'momentum_lookback_days'   : [40, 60, 120, 180],
    'riskparity_lookback_days' : [40, 60, 120],
    'n_top'                    : [3, 5, 8],
    'use_acceleration'         : [True, False],
    'momentum_weight'          : [0.5, 0.7, 1.0],
    'filter_ema'               : [True, False],
    'filter_volatility'        : [True, False],
    'filter_min_momentum'      : [True, False],
}
_ptf = {'stocks_data': _eu_data, 'init_cash': 100_000}

# ── Run X: metric=CAGR ───────────────────────────────────────────────────────
print('--- Run X: metric=CAGR ---')
_, diag_cagr = reduce_grid_via_stability(
    ptf_config=_ptf, full_grid=param_grid_v3,
    full_start_date='2015-01-01', full_end_date='2024-12-31',
    metric='CAGR', k=3, verbose=False,
)
print('Done')

# ── Run Y: metric=Sharpe ─────────────────────────────────────────────────────
print('--- Run Y: metric=Sharpe ---')
_, diag_sharpe = reduce_grid_via_stability(
    ptf_config=_ptf, full_grid=param_grid_v3,
    full_start_date='2015-01-01', full_end_date='2024-12-31',
    metric='Sharpe', k=3, verbose=False,
)
print('Done')

# ── Side-by-side comparison table ────────────────────────────────────────────
print()
print('=== Stability recommendation comparison: CAGR vs Sharpe ===')
header = f"{'flag':<25} {'CAGR_delta':>11} {'CAGR_coh':>9} {'Sharpe_delta':>13} {'Sharpe_coh':>11} {'reco_CAGR':>10} {'reco_Sharpe':>12} {'DIVERGE?':>9}"
print(header)
print('-' * len(header))
_flags = ['filter_ema', 'filter_min_momentum', 'filter_volatility', 'use_acceleration']
n_diverge = 0
for flag in _flags:
    rc = diag_cagr.loc[diag_cagr.flag_name == flag].iloc[0]
    rs = diag_sharpe.loc[diag_sharpe.flag_name == flag].iloc[0]
    c_delta = f'{rc.mean_delta:+.4f}' if not (isinstance(rc.mean_delta, float) and math.isnan(rc.mean_delta)) else 'NaN'
    s_delta = f'{rs.mean_delta:+.4f}' if not (isinstance(rs.mean_delta, float) and math.isnan(rs.mean_delta)) else 'NaN'
    c_coh = str(rc.coherent_sign) if rc.coherent_sign is not None else 'n/a'
    s_coh = str(rs.coherent_sign) if rs.coherent_sign is not None else 'n/a'
    c_reco = str(rc.recommended_value) if rc.recommended_value is not None else 'n/a'
    s_reco = str(rs.recommended_value) if rs.recommended_value is not None else 'n/a'
    diverge = rc.recommended_value != rs.recommended_value and rc.evaluated and rs.evaluated
    if diverge: n_diverge += 1
    div_str = '*** YES ***' if diverge else 'no'
    print(f'{flag:<25} {c_delta:>11} {c_coh:>9} {s_delta:>13} {s_coh:>11} {c_reco:>10} {s_reco:>12} {div_str:>9}')

print()
print(f'Flags with divergent recommendation: {n_diverge} / {len(_flags)}')
print()

# ── use_acceleration detail ───────────────────────────────────────────────────
print('--- use_acceleration detail ---')
ra_c = diag_cagr.loc[diag_cagr.flag_name == 'use_acceleration'].iloc[0]
ra_s = diag_sharpe.loc[diag_sharpe.flag_name == 'use_acceleration'].iloc[0]
print(f'  CAGR:  delta_per_period = {ra_c.delta_per_period}')
print(f'  Sharpe: delta_per_period = {ra_s.delta_per_period}')
print(f'  CAGR  reco={ra_c.recommended_value}, note={ra_c.diagnostic_note}')
print(f'  Sharpe reco={ra_s.recommended_value}, note={ra_s.diagnostic_note}')

# ── Grid cardinality ─────────────────────────────────────────────────────────
print()
_orig = _prod(len(v) for v in param_grid_v3.values())
_ev_c = [r for _, r in diag_cagr.iterrows() if r.evaluated and r.recommended_value is not None]
_ev_s = [r for _, r in diag_sharpe.iterrows() if r.evaluated and r.recommended_value is not None]
_red_c = _orig
_red_s = _orig
for r in _ev_c: _red_c //= 2
for r in _ev_s: _red_s //= 2
print(f'Grid cardinality: original={_orig}')
print(f'  After CAGR stability:   {_red_c}  ({_orig/_red_c:.1f}x)')
print(f'  After Sharpe stability: {_red_s}  ({_orig/_red_s:.1f}x)')

In [ ]:
# =============================================================================
# MILESTONE 2 — Overfitting Check smoke test: satellite vs core
# =============================================================================
# Aspettative pre-dichiarate:
#   S2: entrambi passano (3/4 flag coerenti, soddisfa threshold 0.50 e 0.75)
#   S3: satellite pass (CAGR) / core incerto (Calmar + p<=0.05 severo)
#   S4: satellite probabile pass (DSR>0) / core probabile fail (DSR>0.5)
#   Esito atteso: promoted satellite=True, core=False (Alpha Euro è satellite)
#
#   n_total_trials=3456 passato esplicitamente (griglia originale pre-riduzione)
#   per penalizzare correttamente S4 DSR.

import yfinance as yf, math
from math import prod as _prod

# ── Data ─────────────────────────────────────────────────────────────────────
_eu_tickers = ['ENEL.MI', 'ISP.MI', 'UCG.MI', 'ENI.MI', 'STLAM.MI', 'PRY.MI', 'G.MI', 'LDO.MI', 'PST.MI', 'SAP.DE', 'SIE.DE', 'IFX.DE', 'DTE.DE', 'BAYN.DE', 'ALV.DE', 'MRK.DE', 'BAS.DE', 'RWE.DE', 'MC.PA', 'OR.PA', 'AIR.PA', 'SAN.PA', 'BNP.PA', 'AI.PA', 'KER.PA', 'HO.PA', 'ML.PA', 'ITX.MC', 'IBE.MC', 'SAN.MC', 'REP.MC', 'TEF.MC', 'ACS.MC', 'ACX.MC', 'CLNX.MC', 'GRF.MC']
_eu_data = yf.download(_eu_tickers, start='2013-01-01', end='2025-01-01',
    auto_adjust=True, progress=False)['Close']
_eu_data = _eu_data.dropna(thresh=int(len(_eu_data)*0.80),axis=1).dropna(how='all').ffill()
_bench = yf.download('^STOXX50E', start='2013-01-01', end='2025-01-01',
    auto_adjust=True, progress=False)['Close'].squeeze()
print(f'Tickers: {_eu_data.shape[1]}, bench: {len(_bench)} rows')

param_grid_v3 = {
    'rebalance_frequency': ['ME','QE'], 'momentum_lookback_days': [40,60,120,180],
    'riskparity_lookback_days': [40,60,120], 'n_top': [3,5,8],
    'use_acceleration': [True,False], 'momentum_weight': [0.5,0.7,1.0],
    'filter_ema': [True,False], 'filter_volatility': [True,False],
    'filter_min_momentum': [True,False],
}
print(f'Original grid: {_prod(len(v) for v in param_grid_v3.values())} combos')

# ── Stability + reduced WFO ───────────────────────────────────────────────────
_ptf_cfg = {'stocks_data': _eu_data, 'init_cash': 100_000}
_red_grid, _stab_report = reduce_grid_via_stability(
    ptf_config=_ptf_cfg, full_grid=param_grid_v3,
    full_start_date='2015-01-01', full_end_date='2024-12-31',
    metric='CAGR', k=3, verbose=False,
)
print(f'Reduced grid: {_prod(len(v) for v in _red_grid.values())} combos')

_wfo_b = walk_forward_rotational(
    stocks_data=_eu_data, benchmark_data=_bench,
    param_grid=_red_grid, ratio='3:1', metric='Sharpe Ratio',
    start_date='2018-01-01', end_date='2024-01-01',
    verbose=False, plot=False, n_jobs=1,
)
print(f'WFO windows: {len(_wfo_b)}')
print(_wfo_b[['rebalance_frequency','momentum_lookback_days','n_top','TestScore']].to_string())
print()

# ── Run 1: profile=satellite ──────────────────────────────────────────────────
print('=== Run 1: profile=satellite ===')
_prom_sat, _rep_sat = overfitting_check_rotational(
    wfo_summary=_wfo_b, stocks_data=_eu_data, benchmark_data=_bench,
    param_grid=_red_grid, n_total_trials=3456,
    profile='satellite', stability_report=_stab_report,
    n_bootstrap=50, seed=42, verbose=True,
)

# ── Run 2: profile=core ───────────────────────────────────────────────────────
print('=== Run 2: profile=core ===')
_prom_core, _rep_core = overfitting_check_rotational(
    wfo_summary=_wfo_b, stocks_data=_eu_data, benchmark_data=_bench,
    param_grid=_red_grid, n_total_trials=3456,
    profile='core', stability_report=_stab_report,
    n_bootstrap=50, seed=42, verbose=True,
)

# ── Comparison table ─────────────────────────────────────────────────────────
print('='*64)
print('SIGNAL COMPARISON: satellite vs core')
print('='*64)
_sigs = ['S1_plateau','S2_coherence','S3_bootstrap','S4_dsr']
_hdr = f"{'Signal':<16} {'sat_val':>10} {'sat_pass':>9} {'core_val':>10} {'core_pass':>9} {'aspettativa'}"
_expected = {'S1_plateau':'pass/pass','S2_coherence':'pass/pass',
             'S3_bootstrap':'sat=pass / core=incerto','S4_dsr':'sat=pass / core=fail'}
print(_hdr)
print('-'*len(_hdr))
for s in _sigs:
    sv = _rep_sat['signals'][s].get('value') or _rep_sat['signals'][s].get('p_value') or _rep_sat['signals'][s].get('dsr')
    cv = _rep_core['signals'][s].get('value') or _rep_core['signals'][s].get('p_value') or _rep_core['signals'][s].get('dsr')
    sp = 'PASS' if _rep_sat['signals'][s]['pass'] else 'FAIL'
    cp = 'PASS' if _rep_core['signals'][s]['pass'] else 'FAIL'
    svs = f'{sv:+.4f}' if isinstance(sv,float) and not math.isnan(sv) else 'NaN'
    cvs = f'{cv:+.4f}' if isinstance(cv,float) and not math.isnan(cv) else 'NaN'
    exp = _expected.get(s,'')
    print(f'{s:<16} {svs:>10} {sp:>9} {cvs:>10} {cp:>9}   {exp}')
print()
print(f'satellite: promoted={_prom_sat}  (signals {_rep_sat["n_signals_passed"]}/{len(_sigs)})')
print(f'core:      promoted={_prom_core}  (signals {_rep_core["n_signals_passed"]}/{len(_sigs)})')
print()
print('Aspettative pre-dichiarate: satellite=True, core=False')
print(f'Confermate: sat={_prom_sat==True}, core={_prom_core==False}')

In [ ]:
# =============================================================================
# DIAGNOSTICA S3 SATELLITE — Lettura A (metrica) vs Lettura B (varianza bootstrap)
# =============================================================================
# Run satellite_cagr_50  (baseline, già eseguito in Cell 80): CAGR, n=50 → p=0.82 FAIL
# Run satellite_sharpe   : metric override='Sharpe Ratio', n=50  → testa Lettura A
# Run satellite_cagr_500 : CAGR default, n=500                   → testa Lettura B
#
# Lettura A confermata se: satellite_sharpe S3 passa (p<=0.10)
# Lettura B confermata se: satellite_cagr_500 p ~= 0.82 (segnale stabile, non artefatto)

import yfinance as yf, math
from math import prod as _prod

_eu_tickers = ['ENEL.MI', 'ISP.MI', 'UCG.MI', 'ENI.MI', 'STLAM.MI', 'PRY.MI', 'G.MI', 'LDO.MI', 'PST.MI', 'SAP.DE', 'SIE.DE', 'IFX.DE', 'DTE.DE', 'BAYN.DE', 'ALV.DE', 'MRK.DE', 'BAS.DE', 'RWE.DE', 'MC.PA', 'OR.PA', 'AIR.PA', 'SAN.PA', 'BNP.PA', 'AI.PA', 'KER.PA', 'HO.PA', 'ML.PA', 'ITX.MC', 'IBE.MC', 'SAN.MC', 'REP.MC', 'TEF.MC', 'ACS.MC', 'ACX.MC', 'CLNX.MC', 'GRF.MC']
_eu_data = yf.download(_eu_tickers, start='2013-01-01', end='2025-01-01',
    auto_adjust=True, progress=False)['Close']
_eu_data = _eu_data.dropna(thresh=int(len(_eu_data)*0.80),axis=1).dropna(how='all').ffill()
_bench = yf.download('^STOXX50E', start='2013-01-01', end='2025-01-01',
    auto_adjust=True, progress=False)['Close'].squeeze()

pgv3 = {'rebalance_frequency':['ME','QE'],'momentum_lookback_days':[40,60,120,180],
    'riskparity_lookback_days':[40,60,120],'n_top':[3,5,8],
    'use_acceleration':[True,False],'momentum_weight':[0.5,0.7,1.0],
    'filter_ema':[True,False],'filter_volatility':[True,False],'filter_min_momentum':[True,False]}
_ptf = {'stocks_data': _eu_data, 'init_cash': 100_000}
_rgrid, _srep = reduce_grid_via_stability(
    ptf_config=_ptf, full_grid=pgv3,
    full_start_date='2015-01-01', full_end_date='2024-12-31',
    metric='CAGR', k=3, verbose=False)
_wfob = walk_forward_rotational(
    stocks_data=_eu_data, benchmark_data=_bench, param_grid=_rgrid,
    ratio='3:1', metric='Sharpe Ratio', start_date='2018-01-01', end_date='2024-01-01',
    verbose=False, plot=False, n_jobs=1)
print(f'Setup done: {_eu_data.shape[1]} tickers, {len(_wfob)} WFO windows')
print()

# ── Run satellite_sharpe (Lettura A: stessa metrica di S3 = Sharpe) ──────────
print('--- Run satellite_sharpe: metric=Sharpe Ratio, n_bootstrap=50 ---')
_psat_sh, _rsat_sh = overfitting_check_rotational(
    wfo_summary=_wfob, stocks_data=_eu_data, benchmark_data=_bench,
    param_grid=_rgrid, n_total_trials=3456,
    profile='satellite', metric='Sharpe Ratio',
    stability_report=_srep, n_bootstrap=50, seed=42, verbose=False)
print(f'  S3 Sharpe: p={_rsat_sh["signals"]["S3_bootstrap"]["p_value"]:.3f}  '  
      f'pass={_rsat_sh["signals"]["S3_bootstrap"]["pass"]}  '  
      f'promoted={_psat_sh}')
print()

# ── Run satellite_cagr_500 (Lettura B: stessa metrica, più bootstrap) ────────
print('--- Run satellite_cagr_500: metric=CAGR (default), n_bootstrap=500 ---')
_psat_500, _rsat_500 = overfitting_check_rotational(
    wfo_summary=_wfob, stocks_data=_eu_data, benchmark_data=_bench,
    param_grid=_rgrid, n_total_trials=3456,
    profile='satellite',
    stability_report=_srep, n_bootstrap=500, seed=42, verbose=False)
print(f'  S3 CAGR n=500: p={_rsat_500["signals"]["S3_bootstrap"]["p_value"]:.3f}  '
      f'pass={_rsat_500["signals"]["S3_bootstrap"]["pass"]}  '
      f'promoted={_psat_500}')
print()

# ── Comparison table ──────────────────────────────────────────────────────────
print('='*72)
print('CONFRONTO S3 — Diagnostica Lettura A (metrica) vs Lettura B (varianza)')
print('='*72)
_orig_p3   = 0.820  # da Cell 80
_orig_prom = True
_runs = [
    ('satellite_cagr_50 (baseline)', 'CAGR',        50,  _orig_p3,  _orig_prom),
    ('satellite_sharpe_50',          'Sharpe Ratio', 50,  _rsat_sh['signals']['S3_bootstrap']['p_value'],  _psat_sh),
    ('satellite_cagr_500',           'CAGR',        500, _rsat_500['signals']['S3_bootstrap']['p_value'], _psat_500),
]
print(f"{'Run':<30} {'S3 metric':<14} {'n_boot':>6} {'p-value':>8} {'S3':>5} {'promoted':>9}")
print('-'*72)
for name, met, nb_, pval, prom in _runs:
    sp = 'PASS' if pval <= 0.10 else 'FAIL'
    print(f"{name:<30} {met:<14} {nb_:>6} {pval:>8.3f} {sp:>5} {str(prom):>9}")
print()

# ── Interpretazione automatica ────────────────────────────────────────────────
_sh_p  = _rsat_sh['signals']['S3_bootstrap']['p_value']
_c500_p = _rsat_500['signals']['S3_bootstrap']['p_value']
print('Interpretazione:')
if _sh_p <= 0.10:
    print(f'  Lettura A CONFERMATA: con Sharpe p={_sh_p:.3f} <= 0.10.')
    print('  Il PTF ha edge su Sharpe; il fallimento CAGR è artefatto della metrica.')
    print('  Suggerimento: override metric=Sharpe Ratio per satellite quando S3 è critico.')
else:
    print(f'  Lettura A NON confermata: anche con Sharpe p={_sh_p:.3f} > 0.10.')
    print('  Il PTF non batte le baseline random neanche su Sharpe.')
if abs(_c500_p - _orig_p3) < 0.10:
    print(f'  Lettura B CONFERMATA: p={_c500_p:.3f} stabile rispetto a baseline {_orig_p3:.3f}.')
    print('  N=50 era sufficiente; il dato non è artefatto bootstrap.')
else:
    print(f'  Lettura B: p={_c500_p:.3f} vs baseline {_orig_p3:.3f} — variazione significativa.')
    print('  Il p-value era instabile con N=50; usare N>=500 per stime affidabili.')